# VertebraPrompt BoxRefiner + MedSAM

Copia experimental enfocada en mejorar la calidad de las cajas antes de MedSAM. Se conserva el detector ganador y se entrena un refinador local que corrige centro y tamano de cada caja usando un crop alrededor de la vertebra.

Esta prueba no intenta renombrar vertebras. La meta es subir el rendimiento flexible y, si la caja queda mejor centrada, mejorar tambien la segmentacion final de MedSAM.

<!-- codex-explicacion -->
Este es el notebook de la estrategia ganadora por metricas promedio en test: VertebraPrompt + BoxRefiner + MedSAM. Parte de lo aprendido con NN-SAM y agrega una representacion auxiliar de vertebras junto con un refinador local de cajas.

La hipotesis fue que MedSAM no necesitaba un cambio radical, sino prompts mas estables. Por eso la mejora se concentra antes de llamar a MedSAM: mejores propuestas, mejor refinamiento y evaluacion estricta/flexible.


## 1. Configuracion general

Esta celda define rutas, semillas, parametros de entrenamiento y opciones de evaluacion. Se concentra todo aqui para que la corrida sea reproducible y para evitar cambios escondidos dentro de funciones.

Se usa CUDA de forma obligatoria porque el flujo entrena una red de prompts y tambien ajusta MedSAM. El notebook escribe todo en `resultados_nn_sam_entrenamiento_completo`, de modo que los resultados de esta prueba no se mezclen con experimentos anteriores.

La configuracion importante fue:

- Entrenar VertebraPrompt-Net desde cero para generar prompts automaticos.
- Entrenar MedSAM dentro de este mismo notebook, no importar el ajuste desde otro archivo.
- Comparar `medsam_general`, `medsam_decoder` y `medsam_decoder_encoder_parcial`.
- Probar si conviene expandir cajas antes de MedSAM.

El resultado posterior mostro que la mejor ruta fue `medsam_decoder_encoder_parcial + box_only + sin_pad`.

<!-- codex-explicacion -->
Centraliza rutas, semilla, CUDA y banderas de ejecucion. Esta celda define si se entrena, se evalua o se reutilizan checkpoints.


In [ ]:
# --- Librerias y reproducibilidad ---
import json
import math
import sys
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- GPU: esta corrida debe usar CUDA para evitar tiempos impracticos ---
REQUIERE_CUDA = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIERE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError("CUDA no esta disponible. Esta prueba esta pensada para GPU.")

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(0))

# --- Rutas de datos y carpeta de resultados ---
DATASET_ROOT = Path("C:/Users/luisf/Downloads/ProyectoFinal/Scoliosis_Dataset")
MEDSAM_DATA_ROOT = Path("C:/Users/luisf/Downloads/ProyectoFinal/dataset_procesado_scoliosis_medsam/medsam")
LABELS_DICT_PATH = DATASET_ROOT / "diccionario_etiquetas_T1_T12_L1_L5.json"
RESULTADOS_DIR = Path("C:/Users/luisf/Downloads/ProyectoFinal/resultados_vertebraprompt_boxrefiner")
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

# --- Hiperparametros de VertebraPrompt-Net ---
IMG_SIZE = 512
BATCH_SIZE = 2
EPOCHS = 60
PACIENCIA = 10
LR = 3e-4
BASE_CH = 32

SIGMA_CENTRO = 4.5
BOX_EXPAND_W = 1.12
BOX_EXPAND_H = 1.12
PRESENCE_THR = 0.35
USAR_PRESENCE_EN_DECODIFICACION = False
MIN_VISIBLE_LABELS = 8
MAX_CANDIDATOS = 120
TOP_PICOS = 90
MIN_DIST_PICOS = 8
Y_MIN_ANATOMICO_MARGEN = 0.06
CRANEO_SCORE_FACTOR = 0.12
MAX_GAP_REL_DY = 2.40

# --- Aumentacion y evaluacion de cajas ---
# Entrenamiento final de cajas: cada epoca re-muestrea augmentaciones distintas.
AUGMENT_REPEATS = 6
NUM_WORKERS = 0  # En Windows/Jupyter es mas estable; subirlo solo si el entorno lo permite.
TARGET_BOX_EXPAND_W = 1.10
TARGET_BOX_EXPAND_H = 1.12

# Evaluacion principal. Val completo mide comportamiento general; los paneles visuales se limitan a casos trazadores.
EVALUAR_TODO_VAL = True
EVALUAR_TEST_FINAL = True
PACIENTES_VISUALIZACION = ["N_12", "N_32", "S_187", "S_130", "S_190", "S_80"]

# --- Entrenamiento y prueba de MedSAM ---
# MedSAM usa cajas automaticas y permite una expansion leve adicional del prompt.
MEDSAM_EXPANSIONES_BBOX = [
    {"nombre": "sin_pad", "frac_x": 0.00, "frac_y": 0.00},
]
MEDSAM_GUARDAR_FIGURAS = True

# Entrenamiento MedSAM dentro de este mismo notebook de entrega.
# `medsam_general` se conserva como baseline sin ajuste; las otras dos variantes se entrenan aqui.
EJECUTAR_ENTRENAMIENTO_MEDSAM = False
MEDSAM_TRAIN_BATCH_SIZE = 2
MEDSAM_TRAIN_WORKERS = 0
MEDSAM_TRAIN_FRAC_X = 0.04
MEDSAM_TRAIN_FRAC_Y = 0.06
MEDSAM_DECODER_EPOCHS = 12
MEDSAM_DECODER_PATIENCIA = 4
MEDSAM_ENCODER_PARCIAL_EPOCHS = 8
MEDSAM_ENCODER_PARCIAL_PATIENCIA = 3
MEDSAM_LR_DECODER = 1e-4
MEDSAM_LR_DECODER_REFINO = 5e-5
MEDSAM_LR_ENCODER = 1e-5
MEDSAM_WEIGHT_DECAY = 1e-4
MEDSAM_PESO_ESCOLIOSIS = 1.25
MEDSAM_PESO_NORMAL = 1.00
MEDSAM_VERTEBRAS_DIFICILES = ["T1", "T2", "T3", "L4", "L5"]
MEDSAM_PESO_VERTEBRA_DIFICIL = 1.15

# --- Decodificacion anatomica de la columna ---
# Ruta principal: 17 centros y etiquetado desde T1.
# La prueba con L5 como ancla queda solo como diagnostico porque desplazo casos buenos.
N_CAJAS_CAMINO = 17
ETIQUETADO_MODO = "top_anchor"  # top_anchor | bottom_anchor | auto_anchor
ESTRATEGIAS_ETIQUETADO = ["top_anchor"]
AUTO_BOTTOM_Y_REL = 0.82
AUTO_BOTTOM_MIN_EXTRA_BOXES = 4

LAMBDA_HEAT = 1.00
LAMBDA_WH = 0.40
LAMBDA_OFF = 0.12
LAMBDA_PRESENCE = 0.00

# Perdidas nuevas: el modelo aprende un heatmap general y 17 heatmaps anatomicos T1-L5.
LAMBDA_CLASS_HEAT = 0.08
USAR_CLASS_HEAT_EN_DECODIFICACION = False
CLASS_HEAT_BLEND = 0.00
CLASS_SHIFT_PENALTY = 0.025
CLASS_OUT_OF_RANGE_PENALTY = 1.50
SHIFTS_DECODIFICACION_NN = list(range(-6, 7))

BASELINE_CAJAS_PATH = Path("C:/Users/luisf/Downloads/ProyectoFinal/resultados_nn_sam_entrenamiento_completo/cajas_nn_sam_entrenamiento_completo_best.pt")
CARGAR_BASELINE_CAJAS = True
CONGELAR_DETECTOR_Y_ENTRENAR_SOLO_IDENTIDAD = True
CKPT_PATH = Path("C:/Users/luisf/Downloads/ProyectoFinal/resultados_vertebraprompt_net_auxiliar/vertebraprompt_net_auxiliar_best.pt")

print("Device:", DEVICE)
print("IMG_SIZE:", IMG_SIZE)


# --- BoxRefiner local ---
USAR_BOX_REFINER = True
APLICAR_BOX_REFINER_A_MEDSAM = True
EJECUTAR_ENTRENAMIENTO_BOX_REFINER = True
BOX_REFINER_SIZE = 192
BOX_REFINER_BATCH_SIZE = 32
BOX_REFINER_EPOCHS = 55
BOX_REFINER_PACIENCIA = 9
BOX_REFINER_LR = 5e-4
BOX_REFINER_WEIGHT_DECAY = 1e-4
BOX_REFINER_CONTEXT_FRAC = 0.85
BOX_REFINER_BLEND = 0.80
BOX_REFINER_MIN_IOU_TARGET = 0.08
BOX_REFINER_MIN_RECALL_TARGET = 0.35
BOX_REFINER_MAX_ABS_DXY = 0.45
BOX_REFINER_MAX_ABS_LOG_SCALE = 0.45
BOX_REFINER_CKPT_PATH = RESULTADOS_DIR / "box_refiner_best.pt"

print("Resultados:", RESULTADOS_DIR)


# Esta version entrena VertebraPrompt-Net y luego separa evaluacion de prompts de segmentacion MedSAM.
NN_SAM_DIR = RESULTADOS_DIR
EJECUTAR_MEDSAM_BASICO = False
MEDSAM_REPO_DIR = Path("C:/Users/luisf/MedSAM")
if MEDSAM_REPO_DIR.exists() and str(MEDSAM_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(MEDSAM_REPO_DIR))

MEDSAM_CKPT_PATH = Path("C:/Users/luisf/MedSAM/work_dir/MedSAM/medsam_vit_b.pth")
MEDSAM_FINETUNED_PATH = Path("C:/Users/luisf/Downloads/ProyectoFinal/best_medsam_spine_maskdecoder.pth")
MEDSAM_DEMO_PATIENT_ID = "S_187"
MEDSAM_DEMO_SPLIT = "val"
MEDSAM_DEMO_USAR_SHIFT = True
MEDSAM_DEMO_MAX_VERTEBRAS = None  # None segmenta todas las vertebras detectadas del caso demo.



# --- Comparativo final de variantes MedSAM ---
# Comparativo final MedSAM: mismos prompts automaticos NN-SAM, distintos pesos MedSAM.
EJECUTAR_COMPARATIVO_MEDSAM = True
MEDSAM_COMPARAR_PATIENT_IDS = ["N_12", "N_32", "S_187", "S_130", "S_190", "S_80"]
MEDSAM_COMPARAR_USAR_SHIFT = True
MEDSAM_COMPARAR_PROMPT_MODE = "box_only"  # box_only fue la ruta que dejo de fragmentar las mascaras.
MEDSAM_DECODER_PATH = Path("C:/Users/luisf/Downloads/ProyectoFinal/resultados_vertebraprompt_net_auxiliar/medsam_decoder_entrenado_vertebraprompt_aux.pt")
MEDSAM_ENCODER_DECODER_PATH = Path("C:/Users/luisf/Downloads/ProyectoFinal/resultados_vertebraprompt_net_auxiliar/medsam_decoder_encoder_parcial_entrenado_vertebraprompt_aux.pt")


## 2. Carga de datos

Aqui se leen las imagenes, mascaras y `prompts.json` desde el dataset ya procesado para MedSAM. El notebook no crea un split nuevo; usa las carpetas ya existentes `train`, `val` y `test`.

Tambien se construye una auditoria de etiquetas reales por imagen. Esto fue clave porque algunas radiografias no tienen todas las vertebras anotadas. Por ejemplo, si una imagen solo tiene 8 etiquetas en GT, no se debe castigar al modelo por segmentar otras vertebras visibles que no fueron anotadas.

Esta decision cambio la forma de interpretar resultados: no buscamos solo una metrica multiclase rigida, sino entender si el modelo localiza y segmenta vertebras utiles para el flujo automatico.

<!-- codex-explicacion -->
Usa las carpetas train/val/test ya procesadas. El notebook no crea un split nuevo; trabaja sobre la particion estratificada congelada.

<!-- codex-normalizacion -->
Este notebook no vuelve a crear el dataset desde cero. Usa la exportaci?n MedSAM generada previamente en el notebook `03`: `dataset_procesado_scoliosis_medsam/medsam/train`, `val` y `test`.

Por eso, la normalizaci?n base ya viene heredada: im?genes a 1024x1024, formato compatible con MedSAM y m?scaras con IDs anat?micos preservados. Encima de eso, `VertebraPrompt + BoxRefiner` aplica una normalizaci?n interna adicional para su red, normalmente resize a `IMG_SIZE` y escalado por percentiles para estabilizar contraste.


In [ ]:
# Lee el diccionario oficial de etiquetas T1-L5.
with open(LABELS_DICT_PATH, "r", encoding="utf-8") as f:
    LABELS_DICT = json.load(f)

MAPEO_ID = {int(k): v for k, v in LABELS_DICT["mascara_multiclase_id_png"].items()}
CLASES_OBJETIVO = [MAPEO_ID[i] for i in range(1, 18)]
N_CLASES = len(CLASES_OBJETIVO)
VERTEBRA_TO_ID = {nombre: idx for idx, nombre in MAPEO_ID.items() if idx != 0}
ID_TO_VERTEBRA = {idx: nombre for nombre, idx in VERTEBRA_TO_ID.items()}

SPLITS = ["train", "val", "test"]
EXTS_IMG = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]


def buscar_archivo_por_stem(carpeta, stem):
    for ext in EXTS_IMG:
        path = carpeta / f"{stem}{ext}"
        if path.exists():
            return path
    return None


def resolver_paths_split(split):
    split_dir = MEDSAM_DATA_ROOT / split
    prompts_path = split_dir / "prompts.json"
    image_dir = next((split_dir / d for d in ["images", "imgs", "image"] if (split_dir / d).is_dir()), None)
    mask_dir = next((split_dir / d for d in ["masks", "labels", "mask", "annotations"] if (split_dir / d).is_dir()), None)
    if image_dir is None or mask_dir is None or not prompts_path.exists():
        raise FileNotFoundError(f"Split incompleto: {split_dir}")
    return {"image_dir": image_dir, "mask_dir": mask_dir, "prompts_path": prompts_path}


def cargar_imagen(path):
    return np.array(Image.open(path).convert("RGB"))


def cargar_mascara(path):
    return np.array(Image.open(path)).astype(np.int32)


# Convierte prompts.json a un diccionario patient_id -> vertebra -> bbox.
def normalizar_prompts(prompts_split):
    out = {}
    for item in prompts_split:
        patient_id = str(item.get("patient_id"))
        sub = {}
        prompts_item = item.get("prompts", {})
        iterable = prompts_item.values() if isinstance(prompts_item, dict) else prompts_item
        for info in iterable:
            if not isinstance(info, dict):
                continue
            vertebra = str(info.get("vertebra", "")).upper()
            if vertebra in CLASES_OBJETIVO and "bbox_xyxy" in info:
                sub[vertebra] = dict(info)
        if sub:
            out[patient_id] = sub
    return out


def resolver_paths_muestra(split, patient_id):
    info = SPLIT_INFO[split]
    path_img = buscar_archivo_por_stem(info["image_dir"], patient_id)
    path_mask = buscar_archivo_por_stem(info["mask_dir"], patient_id)
    if path_img is None or path_mask is None:
        raise FileNotFoundError(f"No encontre imagen/mascara para {patient_id}")
    return path_img, path_mask


def vertebras_presentes_en_mascara(mask):
    ids = sorted(int(v) for v in np.unique(mask) if 1 <= int(v) <= N_CLASES)
    return [ID_TO_VERTEBRA[i] for i in ids]


def filtrar_prompts_por_gt(prompts_sample, labels_gt):
    permitidas = set(labels_gt)
    return {v: info for v, info in prompts_sample.items() if v in permitidas}


# Audita que vertebras existen realmente en la mascara de cada imagen.
def construir_gt_labels_dicc():
    dicc = {split: {} for split in SPLITS}
    filas = []
    for split in SPLITS:
        for patient_id in sorted(PROMPTS_DICC[split].keys()):
            _, path_mask = resolver_paths_muestra(split, patient_id)
            labels_gt = vertebras_presentes_en_mascara(cargar_mascara(path_mask))
            labels_prompt = sorted(PROMPTS_DICC[split][patient_id].keys(), key=lambda v: VERTEBRA_TO_ID[v])
            extras_prompt = [v for v in labels_prompt if v not in labels_gt]
            faltantes_prompt = [v for v in labels_gt if v not in labels_prompt]
            dicc[split][patient_id] = labels_gt
            filas.append({
                "split": split,
                "patient_id": patient_id,
                "n_gt": len(labels_gt),
                "n_prompts_json": len(labels_prompt),
                "n_prompts_extra_vs_gt": len(extras_prompt),
                "n_gt_sin_prompt_json": len(faltantes_prompt),
                "labels_gt": ",".join(labels_gt),
                "labels_prompt_extra_vs_gt": ",".join(extras_prompt),
                "labels_gt_sin_prompt_json": ",".join(faltantes_prompt),
            })
    return dicc, pd.DataFrame(filas)


SPLIT_INFO = {split: resolver_paths_split(split) for split in SPLITS}
PROMPTS = {}
for split in SPLITS:
    with open(SPLIT_INFO[split]["prompts_path"], "r", encoding="utf-8") as f:
        PROMPTS[split] = json.load(f)

PROMPTS_DICC = {split: normalizar_prompts(PROMPTS[split]) for split in SPLITS}
GT_LABELS_DICC, DF_AUDITORIA_GT = construir_gt_labels_dicc()

for split in SPLITS:
    print(f"{split}: {len(PROMPTS_DICC[split])} pacientes")
print("Clases:", CLASES_OBJETIVO)
display(
    DF_AUDITORIA_GT.groupby("split", as_index=False)
    .agg(
        pacientes=("patient_id", "count"),
        n_gt_promedio=("n_gt", "mean"),
        prompts_extra_vs_gt=("n_prompts_extra_vs_gt", "sum"),
        gt_sin_prompt_json=("n_gt_sin_prompt_json", "sum"),
    )
)


## 3. Plantilla de tamano

La red predice el tamano de cada caja, pero se calcula una plantilla mediana desde `train` como respaldo. Esta plantilla evita cajas absurdas cuando una prediccion local de ancho o alto sale inestable.

La idea no es imponer una anatomia fija, sino tener un limite razonable. Como las vertebras tienen tamanos relativamente consistentes dentro de la escala normalizada, esta plantilla ayuda a estabilizar la inferencia sin reemplazar lo que aprende la red.

<!-- codex-explicacion -->
La plantilla usa train para aprender tamanos vertebrales esperados. BoxRefiner puede ajustar localmente, pero esta referencia evita predicciones absurdas.

<!-- codex-normalizacion -->
La plantilla de tama?o solo tiene sentido porque las im?genes ya est?n en una escala com?n heredada de la exportaci?n MedSAM. Si cada radiograf?a estuviera en una resoluci?n distinta, esta plantilla mezclar?a anatom?a con cambios de escala.


In [ ]:
def construir_template_bbox_train(prompts_dicc, split_template="train"):
    filas = []
    for patient_id, prompts_sample in prompts_dicc[split_template].items():
        for vertebra, info in prompts_sample.items():
            x0, y0, x1, y1 = info["bbox_xyxy"]
            filas.append({
                "patient_id": patient_id,
                "vertebra": vertebra,
                "id_real": VERTEBRA_TO_ID[vertebra],
                "cx_rel": ((x0 + x1) / 2) / 1024,
                "cy_rel": ((y0 + y1) / 2) / 1024,
                "w_rel": (x1 - x0) / 1024,
                "h_rel": (y1 - y0) / 1024,
            })

    df = pd.DataFrame(filas)
    template = (
        df.groupby(["vertebra", "id_real"], as_index=False)
        .agg(cx_rel=("cx_rel", "median"), cy_rel=("cy_rel", "median"), w_rel=("w_rel", "median"), h_rel=("h_rel", "median"))
        .sort_values("id_real")
        .reset_index(drop=True)
    )
    return template, df


template_bbox, df_template = construir_template_bbox_train(PROMPTS_DICC)
display(template_bbox)


## 4. Targets VertebraPrompt-Net

La red de prompts sigue una idea tipo CenterNet: no predice una mascara, sino centros vertebrales, ancho/alto y offset subpixel. Esto la vuelve ligera y adecuada como paso previo a MedSAM.

En esta version se usaron augmentaciones repetidas por epoca (`AUGMENT_REPEATS`) para que el modelo vea variaciones de contraste, ruido, crop y flip. Tambien se entreno con cajas objetivo ligeramente expandidas. Esa expansion no busca inflar artificialmente la metrica de caja, sino producir prompts mas tolerantes para MedSAM.

Lo que se probo antes mostro que el problema no era solo encontrar una curva, sino generar cajas suficientemente centradas y consistentes para que MedSAM no segmente tejido vecino.

<!-- codex-explicacion -->
Los targets incluyen informacion de centros, tamanos, offsets y senales por vertebra. La idea es dar a la red mas contexto anatomico que en CenterNet-lite basico.


In [ ]:
# Expande cajas en el espacio original 1024x1024 antes de crear targets.
def expandir_bbox_1024(bbox, factor_w=1.0, factor_h=1.0):
    x0, y0, x1, y1 = [float(v) for v in bbox]
    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2
    bw = max(1.0, (x1 - x0) * float(factor_w))
    bh = max(1.0, (y1 - y0) * float(factor_h))
    return [
        float(np.clip(cx - bw / 2, 0, 1023)),
        float(np.clip(cy - bh / 2, 0, 1023)),
        float(np.clip(cx + bw / 2, 0, 1023)),
        float(np.clip(cy + bh / 2, 0, 1023)),
    ]


def transformar_prompts(prompts_sample, crop_y0=0, crop_y1=1024, flip=False):
    crop_h = max(crop_y1 - crop_y0, 1)
    scale_y = 1024 / crop_h
    out = {}

    for vertebra, info in prompts_sample.items():
        x0, y0, x1, y1 = [float(v) for v in info["bbox_xyxy"]]

        if flip:
            x0, x1 = 1023 - x1, 1023 - x0

        y0 = (y0 - crop_y0) * scale_y
        y1 = (y1 - crop_y0) * scale_y
        cy = (y0 + y1) / 2

        if cy < 0 or cy > 1023:
            continue

        x0 = float(np.clip(x0, 0, 1023))
        x1 = float(np.clip(x1, 0, 1023))
        y0 = float(np.clip(y0, 0, 1023))
        y1 = float(np.clip(y1, 0, 1023))

        if x1 - x0 < 4 or y1 - y0 < 4:
            continue

        nuevo = dict(info)
        nuevo["bbox_xyxy"] = [x0, y0, x1, y1]
        out[vertebra] = nuevo

    return out


def elegir_crop_vertical(prompts_sample, p_crop=0.40):
    if random.random() > p_crop:
        return 0, 1024

    centros = []
    for info in prompts_sample.values():
        x0, y0, x1, y1 = info["bbox_xyxy"]
        centros.append((y0 + y1) / 2)
    if len(centros) < 5:
        return 0, 1024

    for _ in range(20):
        crop_h = random.randint(680, 1024)
        y0 = random.randint(0, 1024 - crop_h)
        y1 = y0 + crop_h
        n_visible = sum(y0 <= c <= y1 for c in centros)
        if n_visible >= 5:
            return y0, y1

    return 0, 1024


# Carga imagen, aplica augmentacion y filtra prompts a etiquetas realmente disponibles.
def cargar_imagen_entrenamiento(split, patient_id, augment=False):
    path_img, _ = resolver_paths_muestra(split, patient_id)
    img = Image.open(path_img).convert("L")
    prompts = PROMPTS_DICC[split][patient_id]
    labels_gt = GT_LABELS_DICC.get(split, {}).get(patient_id, list(prompts.keys()))
    prompts_gt = filtrar_prompts_por_gt(prompts, labels_gt)
    if prompts_gt:
        prompts = prompts_gt

    crop_y0, crop_y1 = elegir_crop_vertical(prompts) if augment else (0, 1024)
    flip = bool(augment and random.random() < 0.50)

    if crop_y0 != 0 or crop_y1 != 1024:
        img = img.crop((0, crop_y0, 1024, crop_y1)).resize((1024, 1024), Image.BILINEAR)
    if flip:
        img = ImageOps.mirror(img)

    img_np = np.array(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    if augment:
        alpha = np.random.uniform(0.88, 1.12)
        beta = np.random.uniform(-0.05, 0.05) * 255
        ruido = np.random.normal(0, np.random.uniform(0, 4), img_np.shape)
        img_np = np.clip(img_np * alpha + beta + ruido, 0, 255)

    p1, p99 = np.percentile(img_np, [1, 99.5])
    img_np = np.clip((img_np - p1) / (p99 - p1 + 1e-6), 0, 1)
    img_t = torch.from_numpy(img_np[None, ...].astype(np.float32))

    prompts_t = transformar_prompts(prompts, crop_y0=crop_y0, crop_y1=crop_y1, flip=flip)
    return img_t, prompts_t


def draw_gaussian(heat, cx, cy, sigma=SIGMA_CENTRO):
    radius = int(max(2, sigma * 3))
    x0 = max(0, int(cx) - radius)
    x1 = min(IMG_SIZE - 1, int(cx) + radius)
    y0 = max(0, int(cy) - radius)
    y1 = min(IMG_SIZE - 1, int(cy) + radius)
    if x1 <= x0 or y1 <= y0:
        return

    yy, xx = np.mgrid[y0:y1 + 1, x0:x1 + 1]
    g = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2)).astype(np.float32)
    heat[y0:y1 + 1, x0:x1 + 1] = np.maximum(heat[y0:y1 + 1, x0:x1 + 1], g)


# Construye heatmap/wh/offset/presencia para entrenamiento tipo CenterNet.

def targets_desde_prompts(prompts_sample):
    """Targets multi-tarea: centro general, centro por clase, caja, offset y presencia."""
    heat = np.zeros((1, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    class_heat = np.zeros((N_CLASES, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    wh = np.zeros((2, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    off = np.zeros((2, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    mask = np.zeros((1, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    presence = np.zeros((N_CLASES,), dtype=np.float32)

    for vertebra, info in prompts_sample.items():
        if vertebra not in VERTEBRA_TO_ID:
            continue
        label_idx = VERTEBRA_TO_ID[vertebra] - 1
        x0, y0, x1, y1 = expandir_bbox_1024(info["bbox_xyxy"], TARGET_BOX_EXPAND_W, TARGET_BOX_EXPAND_H)
        cx = ((x0 + x1) / 2) / 1024 * IMG_SIZE
        cy = ((y0 + y1) / 2) / 1024 * IMG_SIZE
        xi = int(np.clip(np.floor(cx), 0, IMG_SIZE - 1))
        yi = int(np.clip(np.floor(cy), 0, IMG_SIZE - 1))

        draw_gaussian(heat[0], cx, cy)
        draw_gaussian(class_heat[label_idx], cx, cy)
        heat[0, yi, xi] = 1.0
        class_heat[label_idx, yi, xi] = 1.0
        mask[0, yi, xi] = 1.0
        wh[0, yi, xi] = np.clip((x1 - x0) / 1024, 0.01, 0.40)
        wh[1, yi, xi] = np.clip((y1 - y0) / 1024, 0.01, 0.40)
        off[0, yi, xi] = cx - xi
        off[1, yi, xi] = cy - yi
        presence[label_idx] = 1.0

    return {
        "heat": torch.from_numpy(heat),
        "class_heat": torch.from_numpy(class_heat),
        "wh": torch.from_numpy(wh),
        "off": torch.from_numpy(off),
        "mask": torch.from_numpy(mask),
        "presence": torch.from_numpy(presence),
    }


# Dataset de prompts: repetir muestras aumenta augmentaciones por epoca sin duplicar archivos.
class VertebraPromptDataset(Dataset):
    def __init__(self, split, augment=False, patient_ids=None, repeat=1):
        self.split = split
        self.augment = augment
        base_ids = list(patient_ids) if patient_ids is not None else sorted(PROMPTS_DICC[split].keys())
        self.samples = [(split, pid) for pid in base_ids for _ in range(max(1, int(repeat)))]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        split, patient_id = self.samples[idx]
        img_t, prompts_t = cargar_imagen_entrenamiento(split, patient_id, augment=self.augment)
        target = targets_desde_prompts(prompts_t)
        return img_t, target, {"split": split, "patient_id": patient_id}


ds_train = VertebraPromptDataset("train", augment=True, repeat=AUGMENT_REPEATS)
ds_val = VertebraPromptDataset("val", augment=False, repeat=1)
print("Train pacientes:", len(PROMPTS_DICC["train"]), "| muestras/epoca:", len(ds_train), "| Val:", len(ds_val))


## 5. Modelo conservador

El modelo de cajas es una U-Net pequena con cabezas tipo CenterNet. Su objetivo es ser rapido y estable, no competir con un detector grande.

Salidas principales:

- `heat`: probabilidad de centro vertebral.
- `wh`: ancho y alto de la caja.
- `off`: ajuste subpixel del centro.
- `presence`: presencia global por nivel anatomico, conservada como diagnostico.

Esta arquitectura fue elegida porque permite entrenar con pocas imagenes y produce una salida interpretable: primero centros, luego ruta anatomica, luego cajas.

<!-- codex-explicacion -->
El modelo sigue siendo liviano porque es un generador de prompts. Se evita una arquitectura enorme para mantener MedSAM como segmentador principal.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class VertebraPromptNet(nn.Module):
    """U-Net ligera multi-tarea: detecta centros, cajas y nombre anatomico T1-L5."""
    def __init__(self, base=BASE_CH):
        super().__init__()
        self.e1 = ConvBlock(1, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.e4 = ConvBlock(base * 4, base * 6)
        self.b = ConvBlock(base * 6, base * 8)

        self.u4 = ConvBlock(base * 8 + base * 6, base * 6)
        self.u3 = ConvBlock(base * 6 + base * 4, base * 4)
        self.u2 = ConvBlock(base * 4 + base * 2, base * 2)
        self.u1 = ConvBlock(base * 2 + base, base)

        # Heat general: conserva sensibilidad para no perder vertebras.
        self.heat_head = nn.Conv2d(base, 1, 1)
        # Heat anatomico: cada canal intenta responder a T1, T2, ..., L5.
        self.class_heat_head = nn.Conv2d(base, N_CLASES, 1)
        self.wh_head = nn.Conv2d(base, 2, 1)
        self.off_head = nn.Conv2d(base, 2, 1)
        self.presence_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base * 8, N_CLASES),
        )

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        e4 = self.e4(F.max_pool2d(e3, 2))
        b = self.b(F.max_pool2d(e4, 2))

        u4 = F.interpolate(b, size=e4.shape[-2:], mode="bilinear", align_corners=False)
        u4 = self.u4(torch.cat([u4, e4], dim=1))
        u3 = F.interpolate(u4, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        u3 = self.u3(torch.cat([u3, e3], dim=1))
        u2 = F.interpolate(u3, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        u2 = self.u2(torch.cat([u2, e2], dim=1))
        u1 = F.interpolate(u2, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        u1 = self.u1(torch.cat([u1, e1], dim=1))

        return {
            "heat": self.heat_head(u1),
            "class_heat": self.class_heat_head(u1),
            "wh": self.wh_head(u1),
            "off": self.off_head(u1),
            "presence": self.presence_head(b),
        }


model = VertebraPromptNet().to(DEVICE)
print("Parametros:", round(sum(p.numel() for p in model.parameters()) / 1e6, 3), "M")


## 6. Entrenamiento auxiliar de identidad anatomica

La red de prompts se entrena desde cero. La perdida principal es focal loss sobre centros, combinada con regresion de ancho/alto y offset. El checkpoint se guarda cuando mejora `val_loss`.

En esta corrida el entrenamiento encontro un buen checkpoint alrededor de la mejor perdida de validacion. Los resultados de cajas mostraron una diferencia importante entre:

- metrica estricta baja, por etiquetas corridas;
- metrica flexible alta, porque las vertebras si se localizan espacialmente.

Eso llevo a una conclusion importante: la red encuentra estructuras vertebrales utiles, pero la asignacion anatomica estricta todavia es el cuello de botella.

<!-- codex-explicacion -->
Esta etapa intenta mejorar la consistencia anatomica sin convertir el problema en una clasificacion aislada. Ayuda a ordenar y nombrar vertebras antes del refinamiento.


In [ ]:
# Focal loss: enfatiza centros verdaderos y reduce falsos positivos del heatmap.
def focal_loss_centernet(logits, target):
    pred = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)
    pos = (target >= 0.999).float()
    neg = (target < 0.999).float()
    neg_weights = torch.pow(1 - target, 4)

    pos_loss = -torch.log(pred) * torch.pow(1 - pred, 2) * pos
    neg_loss = -torch.log(1 - pred) * torch.pow(pred, 2) * neg_weights * neg
    num_pos = pos.sum().clamp(min=1.0)
    return (pos_loss.sum() + neg_loss.sum()) / num_pos


def regression_loss_at_centers(pred_logits, target, mask, kind="sigmoid_l1"):
    pred = torch.sigmoid(pred_logits) if kind == "sigmoid_l1" else pred_logits
    mask2 = mask.expand_as(target)
    denom = mask2.sum().clamp(min=1.0)
    return (torch.abs(pred - target) * mask2).sum() / denom




def class_identity_loss_at_centers(class_logits, class_target, mask):
    """Aprende identidad anatomica solo en centros GT para evitar desbordes del mapa denso."""
    logits = class_logits.float()
    target = class_target.float()
    center_mask = mask.float().expand_as(target)
    denom = center_mask.sum().clamp(min=1.0)
    loss = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    return (loss * center_mask).sum() / denom


def loss_batch(outputs, target):
    """Perdida conservadora: solo entrena identidad auxiliar, sin tocar localizacion."""
    with torch.no_grad():
        heat_loss = focal_loss_centernet(outputs["heat"].float(), target["heat"].float())
        wh_loss = regression_loss_at_centers(outputs["wh"].float(), target["wh"].float(), target["mask"].float(), kind="sigmoid_l1")
        off_loss = regression_loss_at_centers(outputs["off"].float(), target["off"].float(), target["mask"].float(), kind="sigmoid_l1")
        presence_loss = F.binary_cross_entropy_with_logits(outputs["presence"].float(), target["presence"].float())

    class_heat_loss = class_identity_loss_at_centers(outputs["class_heat"], target["class_heat"], target["mask"])
    total = LAMBDA_CLASS_HEAT * class_heat_loss
    return total, {
        "heat": float(heat_loss.detach().cpu()),
        "class_heat": float(class_heat_loss.detach().cpu()),
        "wh": float(wh_loss.detach().cpu()),
        "off": float(off_loss.detach().cpu()),
        "presence": float(presence_loss.detach().cpu()),
    }


def mover_target_device(target):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in target.items()}


def crear_loaders():
    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    val_loader = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    return train_loader, val_loader


def torch_load_compatible(path, map_location=None):
    """Compatible con PyTorch reciente, donde weights_only puede bloquear checkpoints comunes."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def cargar_detector_baseline_en_vertebraprompt():
    """Carga la red de cajas ganadora anterior en las capas compartidas de VertebraPrompt-Net."""
    if not CARGAR_BASELINE_CAJAS:
        return {"baseline_cargado": False, "missing": [], "unexpected": []}
    if not BASELINE_CAJAS_PATH.exists():
        raise FileNotFoundError(f"No existe baseline de cajas: {BASELINE_CAJAS_PATH}")
    state = torch_load_compatible(BASELINE_CAJAS_PATH, map_location=DEVICE)
    state_dict = state.get("model", state)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Baseline de cajas cargado:", BASELINE_CAJAS_PATH)
    print("Faltantes esperados:", missing)
    print("Inesperados:", unexpected)
    return {"baseline_cargado": True, "missing": list(missing), "unexpected": list(unexpected)}


def configurar_entrenamiento_auxiliar():
    """Congela el detector y deja entrenable solo la rama anatomica auxiliar."""
    if not hasattr(model, "class_heat_head"):
        raise RuntimeError(
            "El objeto `model` no tiene class_heat_head. Reinicia el kernel y ejecuta el notebook desde la celda 1, "
            "o vuelve a ejecutar la celda del modelo VertebraPromptNet antes de esta celda."
        )
    if not CONGELAR_DETECTOR_Y_ENTRENAR_SOLO_IDENTIDAD:
        for p in model.parameters():
            p.requires_grad = True
        return
    for p in model.parameters():
        p.requires_grad = False
    for p in model.class_heat_head.parameters():
        p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Parametros entrenables: {n_train:,} / {n_total:,} ({100*n_train/n_total:.3f}%)")


@torch.no_grad()
def evaluar_loss_loader(loader):
    model.eval()
    losses = []
    partes = []
    for x, target, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        target = mover_target_device(target)
        outputs = model(x)
        loss, parts = loss_batch(outputs, target)
        losses.append(float(loss.detach().cpu()))
        partes.append(parts)
    if not losses:
        return np.nan, {}
    mean_parts = {k: float(np.mean([p[k] for p in partes])) for k in partes[0].keys()}
    return float(np.mean(losses)), mean_parts


def entrenar_modelo():
    info_base = cargar_detector_baseline_en_vertebraprompt()
    configurar_entrenamiento_auxiliar()
    train_loader, val_loader = crear_loaders()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    best_val = np.inf
    sin_mejora = 0
    hist = []
    t0 = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for x, target, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            x = x.to(DEVICE, non_blocking=True)
            target = mover_target_device(target)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=False):
                outputs = model(x)
                loss, parts = loss_batch(outputs, target)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss no finita en epoch {epoch}: {loss.item()}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.detach().cpu()))

        val_loss, val_parts = evaluar_loss_loader(val_loader)
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "val_loss": val_loss,
            "val_heat": val_parts.get("heat", np.nan),
            "val_class_heat": val_parts.get("class_heat", np.nan),
            "val_wh": val_parts.get("wh", np.nan),
            "val_off": val_parts.get("off", np.nan),
            "val_presence": val_parts.get("presence", np.nan),
            "tiempo_min": (time.perf_counter() - t0) / 60,
        }
        hist.append(row)
        print(f"epoch={epoch:03d} train={row['train_loss']:.5f} val={val_loss:.5f} heat={row['val_heat']:.4f} class={row['val_class_heat']:.4f} wh={row['val_wh']:.4f}")

        if np.isfinite(val_loss) and val_loss < best_val:
            best_val = val_loss
            sin_mejora = 0
            CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
            torch.save({
                "model": model.state_dict(),
                "config": {
                    "arquitectura": "VertebraPromptNet_auxiliar_conservador",
                    "baseline_cajas_path": str(BASELINE_CAJAS_PATH),
                    "IMG_SIZE": IMG_SIZE,
                    "N_CLASES": N_CLASES,
                    "BASE_CH": BASE_CH,
                    "LAMBDA_CLASS_HEAT": LAMBDA_CLASS_HEAT,
                    "USAR_CLASS_HEAT_EN_DECODIFICACION": USAR_CLASS_HEAT_EN_DECODIFICACION,
                    "info_base": info_base,
                },
            }, CKPT_PATH)
        else:
            sin_mejora += 1
            if sin_mejora >= PACIENCIA:
                print("Early stopping.")
                break

    hist = pd.DataFrame(hist)
    hist.to_csv(RESULTADOS_DIR / "historial_entrenamiento_vertebraprompt_auxiliar.csv", index=False)
    return hist


# Esta version entrena rapido porque parte del detector ganador y solo ajusta la rama anatomica auxiliar.
ENTRENAR_DESDE_CERO = False

if ENTRENAR_DESDE_CERO:
    hist_entrenamiento = entrenar_modelo()
    display(hist_entrenamiento.tail())
else:
    hist_entrenamiento = pd.DataFrame()

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"No existe el checkpoint de prompts: {CKPT_PATH}")

ckpt = torch_load_compatible(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()
print("Checkpoint VertebraPrompt auxiliar cargado:", CKPT_PATH)


## 7. Decodificacion anatomica conservadora

El heatmap entrega muchos candidatos posibles. Esta seccion selecciona una ruta vertebral coherente usando programacion dinamica. La ruta favorece puntos confiables y penaliza saltos verticales/anatomicos improbables.

La asignacion principal es `top_anchor`: se etiquetan las cajas de arriba hacia abajo. Esta estrategia fue la mas estable en pruebas previas, especialmente en imagenes completas. Sin embargo, en escoliosis y en imagenes parciales puede aparecer un desfase: la columna se sigue bien, pero los nombres T/L quedan corridos.

Por eso el notebook conserva la evaluacion de `shift` y la metrica flexible.

<!-- codex-explicacion -->
La decodificacion aplica restricciones anatomicas y ordenamiento. Es conservadora porque los errores de etiqueta afectan mucho la metrica estricta.


In [ ]:
# Convierte una radiografia RGB a tensor normalizado para inferencia NN-SAM.
def imagen_inferencia_tensor(img_rgb):
    gray = np.array(Image.fromarray(img_rgb).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    p1, p99 = np.percentile(gray, [1, 99.5])
    gray = np.clip((gray - p1) / (p99 - p1 + 1e-6), 0, 1)
    return torch.from_numpy(gray[None, None, ...].astype(np.float32)).to(DEVICE)


@torch.no_grad()

def predecir_outputs(img_rgb):
    model.eval()
    x = imagen_inferencia_tensor(img_rgb)
    out = model(x)
    return {
        "heat": torch.sigmoid(out["heat"])[0, 0].detach().cpu().numpy(),
        "class_heat": torch.sigmoid(out["class_heat"])[0].detach().cpu().numpy(),
        "wh": torch.sigmoid(out["wh"])[0].detach().cpu().numpy(),
        "off": torch.sigmoid(out["off"])[0].detach().cpu().numpy(),
        "presence": torch.sigmoid(out["presence"])[0].detach().cpu().numpy(),
    }


# Extrae maximos locales del heatmap como candidatos de centros vertebrales.
def extraer_picos(score_map, n_picos=TOP_PICOS, min_dist=MIN_DIST_PICOS, thr_rel=0.12):
    work = score_map.astype(np.float32).copy()
    picos = []
    max0 = float(work.max())
    if max0 <= 0:
        return picos
    thr = max(max0 * thr_rel, float(np.percentile(work, 90)))

    for _ in range(n_picos):
        idx = int(np.argmax(work))
        y, x = np.unravel_index(idx, work.shape)
        score = float(work[y, x])
        if score < thr:
            break
        picos.append({"x": int(x), "y": int(y), "score": score})
        y0 = max(0, y - min_dist)
        y1 = min(work.shape[0], y + min_dist + 1)
        x0 = max(0, x - min_dist)
        x1 = min(work.shape[1], x + min_dist + 1)
        work[y0:y1, x0:x1] = -np.inf
    return picos


def estimar_y_min_anatomico(img_rgb):
    """Estima un limite superior para penalizar craneo/cuello sin usar GT."""
    gray = np.array(Image.fromarray(img_rgb).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)).astype(np.float32)
    valores = gray[gray > 0]
    if len(valores) == 0:
        return 0.0, {"motivo": "sin_valores"}

    thr = max(5.0, float(np.percentile(valores, 8)))
    active = (gray > thr).astype(np.float32)
    width = active.mean(axis=1)
    kernel = np.ones(21, dtype=np.float32) / 21
    width_s = np.convolve(width, kernel, mode="same")

    h = IMG_SIZE
    top_med = float(np.median(width_s[: int(0.16 * h)]))
    search0 = int(0.12 * h)
    search1 = int(0.55 * h)
    search = width_s[search0:search1]
    if len(search) == 0:
        return 0.0, {"motivo": "sin_search"}

    mid_p85 = float(np.percentile(search, 85))
    umbral_ensanche = max(0.24, top_med * 1.35)
    if top_med > 0.30 or mid_p85 < umbral_ensanche:
        return 0.0, {"motivo": "sin_craneo_claro", "top_med": top_med, "mid_p85": mid_p85}

    idx = np.where(search > umbral_ensanche)[0]
    if len(idx) == 0:
        return 0.0, {"motivo": "sin_ensanche", "top_med": top_med, "mid_p85": mid_p85}

    y_ensanche = float(search0 + idx[0])
    y_min = max(0.0, y_ensanche - Y_MIN_ANATOMICO_MARGEN * h)
    return y_min, {"motivo": "craneo_probable", "top_med": top_med, "mid_p85": mid_p85, "y_ensanche": y_ensanche}


# Selecciona una ruta vertical coherente de candidatos usando programacion dinamica.
def seleccionar_camino_dp(candidatos, n_pasos=N_CAJAS_CAMINO, max_gap_rel=MAX_GAP_REL_DY):
    if len(candidatos) == 0:
        raise RuntimeError("No hay candidatos de centro.")

    candidatos = sorted(candidatos, key=lambda c: (c["y"], c["x"]))[:MAX_CANDIDATOS]
    n = len(candidatos)
    k = min(int(n_pasos), n)
    if k <= 0:
        raise RuntimeError("No hay suficientes candidatos.")

    xs = np.array([c["x"] for c in candidatos], dtype=np.float32)
    ys = np.array([c["y"] for c in candidatos], dtype=np.float32)
    sc = np.array([c["score"] for c in candidatos], dtype=np.float32)

    template = template_bbox.sort_values("id_real").reset_index(drop=True)
    cy_template = template["cy_rel"].to_numpy(dtype=np.float32) * IMG_SIZE
    dy_template = np.diff(cy_template)
    dy_default = float(np.median(dy_template))

    dp = np.full((k, n), -1e9, dtype=np.float32)
    prev = np.full((k, n), -1, dtype=np.int32)
    dp[0] = sc

    for j in range(1, k):
        expected_dy = float(dy_template[j - 1]) if j - 1 < len(dy_template) else dy_default
        expected_dy = max(expected_dy, 4.0)
        min_gap = max(3.0, expected_dy * 0.35)
        max_gap = max(min_gap + 1.0, expected_dy * max_gap_rel)

        for i in range(n):
            dy = ys[i] - ys[:i]
            valid = (dy >= min_gap) & (dy <= max_gap)
            if not valid.any():
                continue
            dx = np.abs(xs[i] - xs[:i])
            dy_pen = np.abs(dy - expected_dy) / expected_dy
            dx_pen = dx / max(IMG_SIZE * 0.22, 1.0)
            trans = dp[j - 1, :i] - 0.34 * dy_pen - 0.08 * dx_pen
            trans[~valid] = -1e9
            best = int(np.argmax(trans))
            dp[j, i] = sc[i] + trans[best]
            prev[j, i] = best

    end = int(np.argmax(dp[k - 1]))
    if dp[k - 1, end] < -1e8 and max_gap_rel < 8.0:
        return seleccionar_camino_dp(candidatos, n_pasos=n_pasos, max_gap_rel=8.0)

    path = [end]
    for j in range(k - 1, 0, -1):
        end = int(prev[j, end])
        if end < 0:
            break
        path.append(end)
    path = path[::-1]

    if len(path) != k:
        orden = np.argsort(sc)[-k:]
        path = sorted(orden.tolist(), key=lambda i: ys[i])

    return [candidatos[i] for i in path]


def bbox_clip(bbox, H, W):
    x0, y0, x1, y1 = [int(round(float(v))) for v in bbox]
    x0, x1 = np.clip([x0, x1], 0, W - 1)
    y0, y1 = np.clip([y0, y1], 0, H - 1)
    if x1 <= x0:
        x1 = min(W - 1, x0 + 1)
    if y1 <= y0:
        y1 = min(H - 1, y0 + 1)
    return [int(x0), int(y0), int(x1), int(y1)]


def decidir_modo_etiquetado_auto(camino):
    if not camino:
        return "top_anchor", {"motivo": "sin_camino"}
    y_bottom_rel = max(float(c["y"]) for c in camino) / IMG_SIZE
    extra_boxes = max(0, len(camino) - N_CLASES)
    if y_bottom_rel >= AUTO_BOTTOM_Y_REL and extra_boxes >= AUTO_BOTTOM_MIN_EXTRA_BOXES:
        return "bottom_anchor", {"motivo": "region_inferior_visible", "y_bottom_rel": y_bottom_rel, "extra_boxes": extra_boxes}
    return "top_anchor", {"motivo": "sin_ancla_inferior_fuerte", "y_bottom_rel": y_bottom_rel, "extra_boxes": extra_boxes}


def elegir_segmento_y_labels(camino, modo_etiquetado="auto_anchor"):
    modo = modo_etiquetado or ETIQUETADO_MODO
    if modo not in ESTRATEGIAS_ETIQUETADO:
        raise ValueError(f"Estrategia de etiquetado no soportada: {modo}")

    camino = sorted(camino, key=lambda c: (c["y"], c["x"]))
    modo_resuelto = modo
    info = {"modo_solicitado": modo}
    if modo == "auto_anchor":
        modo_resuelto, info_auto = decidir_modo_etiquetado_auto(camino)
        info.update(info_auto)

    n_use = min(N_CLASES, len(camino))
    if modo_resuelto == "bottom_anchor":
        segmento = camino[-n_use:]
        labels = list(range(N_CLASES - n_use, N_CLASES))
    else:
        segmento = camino[:n_use]
        labels = list(range(n_use))

    info["modo_resuelto"] = modo_resuelto
    info["n_camino"] = len(camino)
    info["n_prompts"] = len(segmento)
    return segmento, labels, info


# Convierte centros seleccionados en cajas etiquetadas T1-L5.
def construir_prompts_desde_segmento(segmento, labels, img_rgb, presence, modo_info):
    H, W = img_rgb.shape[:2]
    template = template_bbox.sort_values("id_real").reset_index(drop=True)
    prompts = {}
    filas = []

    for orden, (cand, label_idx) in enumerate(zip(segmento, labels)):
        vertebra = ID_TO_VERTEBRA[label_idx + 1]
        trow = template[template["id_real"] == label_idx + 1].iloc[0]
        cx = cand["x"] / IMG_SIZE * W
        cy = cand["y"] / IMG_SIZE * H
        pred_w = np.clip(cand["wh_rel"][0], 0.03, 0.28) * W
        pred_h = np.clip(cand["wh_rel"][1], 0.025, 0.18) * H
        tpl_w = float(trow["w_rel"] * W)
        tpl_h = float(trow["h_rel"] * H)
        bw = (0.65 * pred_w + 0.35 * tpl_w) * BOX_EXPAND_W
        bh = (0.65 * pred_h + 0.35 * tpl_h) * BOX_EXPAND_H
        bbox = bbox_clip([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], H, W)

        prompts[vertebra] = {
            "vertebra": vertebra,
            "id_real": label_idx + 1,
            "bbox_xyxy": bbox,
            "confianza_nn": float(cand["score"]),
            "confianza_original": float(cand.get("score_original", cand["score"])),
            "presencia_nn": float(presence[label_idx]) if label_idx < len(presence) else np.nan,
            "class_heat_label_score": float(cand.get("class_scores", [np.nan] * N_CLASES)[label_idx]) if "class_scores" in cand else np.nan,
            "class_heat_best_label": cand.get("class_best_label", ""),
            "class_heat_best_score": float(cand.get("class_best_score", np.nan)),
            "estrategia_etiquetado": modo_info["modo_solicitado"],
            "estrategia_resuelta": modo_info["modo_resuelto"],
            "orden_caja": int(orden),
            "n_camino": int(modo_info["n_camino"]),
            "y_min_anatomico": float(cand.get("y_min_anatomico", 0.0)),
            "prompt_origen": "vertebraprompt_net",
        }
        filas.append({
            "vertebra": vertebra,
            "id_real": label_idx + 1,
            "cx": cx,
            "cy": cy,
            "confianza": float(cand["score"]),
            "confianza_original": float(cand.get("score_original", cand["score"])),
            "presencia": float(presence[label_idx]) if label_idx < len(presence) else np.nan,
            "class_heat_label_score": float(cand.get("class_scores", [np.nan] * N_CLASES)[label_idx]) if "class_scores" in cand else np.nan,
            "class_heat_best_label": cand.get("class_best_label", ""),
            "class_heat_best_score": float(cand.get("class_best_score", np.nan)),
            "shift_nn_class_heat": int(modo_info.get("shift_nn_class_heat", 0)),
            "penalizado_craneo": bool(cand.get("penalizado_craneo", False)),
            "y_min_anatomico": float(cand.get("y_min_anatomico", 0.0)),
            "bbox_xyxy": bbox,
            "estrategia_etiquetado": modo_info["modo_solicitado"],
            "estrategia_resuelta": modo_info["modo_resuelto"],
            "orden_caja": int(orden),
            "n_camino": int(modo_info["n_camino"]),
        })

    return prompts, pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)



def score_map_vertebraprompt(heat, class_heat):
    """Conservador: los centros salen del heatmap general, no de los mapas por clase."""
    return heat


def refinar_segmento_y_labels_con_class_heat(segmento, labels_base, modo_info):
    """La identidad anatomica queda como diagnostico; no corrige etiquetas por defecto."""
    info = dict(modo_info)
    info["class_heat_usado_para_shift"] = bool(USAR_CLASS_HEAT_EN_DECODIFICACION)
    if not USAR_CLASS_HEAT_EN_DECODIFICACION:
        return segmento, labels_base, info

    mejores = []
    for shift in SHIFTS_DECODIFICACION_NN:
        score = 0.0
        n_valid = 0
        for cand, base_idx in zip(segmento, labels_base):
            new_idx = int(base_idx) - int(shift)
            class_scores = np.asarray(cand.get("class_scores", np.zeros(N_CLASES)), dtype=np.float32)
            if 0 <= new_idx < N_CLASES:
                score += math.log(float(class_scores[new_idx]) + 1e-4)
                n_valid += 1
            else:
                score -= CLASS_OUT_OF_RANGE_PENALTY
        score = score / max(n_valid, 1) - abs(int(shift)) * CLASS_SHIFT_PENALTY
        mejores.append({"shift_nn": int(shift), "score_shift_nn": float(score), "n_valid": int(n_valid)})

    df = pd.DataFrame(mejores).sort_values(["score_shift_nn", "n_valid"], ascending=[False, False])
    best = df.iloc[0].to_dict()
    shift = int(best["shift_nn"])
    segmento_out, labels_out = [], []
    for cand, base_idx in zip(segmento, labels_base):
        new_idx = int(base_idx) - shift
        if 0 <= new_idx < N_CLASES:
            segmento_out.append(cand)
            labels_out.append(new_idx)
    info["shift_nn_class_heat"] = shift
    info["score_shift_nn"] = float(best["score_shift_nn"])
    info["n_prompts_tras_shift_nn"] = len(labels_out)
    return segmento_out, labels_out, info


def prompts_desde_outputs(img_rgb, out, modo_etiquetado=None):
    heat, class_heat = out["heat"], out.get("class_heat")
    wh_map, off_map, presence = out["wh"], out["off"], out["presence"]
    score_map = score_map_vertebraprompt(heat, class_heat)
    picos = extraer_picos(score_map)
    y_min_hm, info_anatomico = estimar_y_min_anatomico(img_rgb)

    candidatos = []
    for p in picos:
        x, y = p["x"], p["y"]
        cx_hm = x + float(off_map[0, y, x])
        cy_hm = y + float(off_map[1, y, x])
        score = float(p["score"])
        class_scores = class_heat[:, y, x].astype(np.float32) if class_heat is not None else np.zeros(N_CLASES, dtype=np.float32)
        best_class_idx = int(np.argmax(class_scores)) if len(class_scores) else -1
        best_class_score = float(class_scores[best_class_idx]) if best_class_idx >= 0 else 0.0

        if cy_hm < y_min_hm:
            if cy_hm < y_min_hm - IMG_SIZE * 0.06:
                continue
            score *= CRANEO_SCORE_FACTOR

        candidatos.append({
            "x": cx_hm,
            "y": cy_hm,
            "score": score,
            "score_original": float(heat[y, x]),
            "score_fusion": float(p["score"]),
            "class_scores": class_scores.tolist(),
            "class_best_idx": best_class_idx,
            "class_best_label": ID_TO_VERTEBRA.get(best_class_idx + 1, ""),
            "class_best_score": best_class_score,
            "penalizado_craneo": bool(cy_hm < y_min_hm),
            "y_min_anatomico": float(y_min_hm),
            "wh_rel": [float(wh_map[0, y, x]), float(wh_map[1, y, x])],
        })

    camino = seleccionar_camino_dp(candidatos, n_pasos=N_CAJAS_CAMINO)
    segmento, labels, modo_info = elegir_segmento_y_labels(camino, modo_etiquetado=modo_etiquetado)
    segmento, labels, modo_info = refinar_segmento_y_labels_con_class_heat(segmento, labels, modo_info)
    prompts, df_centros = construir_prompts_desde_segmento(segmento, labels, img_rgb, presence, modo_info)

    df_candidatos = pd.DataFrame(candidatos)
    df_candidatos.attrs["info_anatomico"] = info_anatomico
    df_candidatos.attrs["modo_info"] = modo_info
    return prompts, df_centros, df_candidatos


def generar_prompts_nn(split, patient_id, modo_etiquetado=None):
    path_img, path_mask = resolver_paths_muestra(split, patient_id)
    img = cargar_imagen(path_img)
    mask = cargar_mascara(path_mask)
    out = predecir_outputs(img)
    prompts, df_centros, df_candidatos = prompts_desde_outputs(img, out, modo_etiquetado=modo_etiquetado)
    return img, mask, prompts, df_centros, df_candidatos, out


## 8. Metricas por etiqueta GT

Esta seccion define como se evalua la calidad de las cajas. Se usan tres lecturas complementarias:

- Estricta: compara `T1` contra `T1`, `T2` contra `T2`, etc. Mide deteccion + asignacion anatomica.
- Flexible: para cada vertebra GT busca la mejor caja generada, sin importar el nombre. Mide si se encontro la vertebra.
- Shift: prueba si un corrimiento uniforme de etiquetas explica parte del error.

La metrica flexible no es una estrategia de produccion porque usa el GT para decidir la mejor correspondencia. Su valor es diagnostico: separa un problema de segmentacion real de un problema de nombramiento anatomico.

<!-- codex-explicacion -->
Se evalua solo contra etiquetas existentes en cada imagen. Esto fue importante para casos como S_80, donde no todas las vertebras estaban anotadas.


In [ ]:
# Metricas geometricas de caja contra mascara vertebral disponible.
def mask_binaria_vertebra(mask, vertebra):
    return (mask == VERTEBRA_TO_ID[vertebra]).astype(np.uint8)


def vertebras_gt_eval(mascara_gt):
    return vertebras_presentes_en_mascara(mascara_gt)


def metricas_bbox_contra_gt(gt, bbox):
    x0, y0, x1, y1 = [int(v) for v in bbox]
    cover = np.zeros_like(gt, dtype=bool)
    cover[y0:y1 + 1, x0:x1 + 1] = True
    inter = int(np.logical_and(gt, cover).sum())
    union = int(np.logical_or(gt, cover).sum())
    bbox_area = int(cover.sum())
    total_gt = int(gt.sum())

    ys, xs = np.where(gt)
    if len(xs) == 0:
        center_error = np.nan
    else:
        cx_gt, cy_gt = xs.mean(), ys.mean()
        cx_box, cy_box = (x0 + x1) / 2, (y0 + y1) / 2
        center_error = math.sqrt((cx_box - cx_gt) ** 2 + (cy_box - cy_gt) ** 2)

    return {
        "bbox_iou": inter / union if union else 0.0,
        "bbox_recall": inter / total_gt if total_gt else 0.0,
        "bbox_precision": inter / bbox_area if bbox_area else 0.0,
        "center_error_px": center_error,
    }


def mejor_prompt_para_gt(prompts_auto, gt):
    mejor = None
    mejor_key = None
    for vertebra_prompt, info in prompts_auto.items():
        if "bbox_xyxy" not in info:
            continue
        met = metricas_bbox_contra_gt(gt, info["bbox_xyxy"])
        center = met["center_error_px"]
        center_key = -center if np.isfinite(center) else -1e9
        key = (met["bbox_iou"], met["bbox_recall"], center_key)
        if mejor is None or key > mejor_key:
            mejor = {
                "prompt_flexible_vertebra": vertebra_prompt,
                "prompt_flexible_id": VERTEBRA_TO_ID.get(vertebra_prompt, np.nan),
                "bbox_iou_flexible": met["bbox_iou"],
                "bbox_recall_flexible": met["bbox_recall"],
                "bbox_precision_flexible": met["bbox_precision"],
                "center_error_flexible_px": met["center_error_px"],
                "confianza_flexible": info.get("confianza_nn", np.nan),
            }
            mejor_key = key
    return mejor


# Evalua cajas estrictas y tambien mejor correspondencia flexible por vertebra GT.
def evaluar_cajas(prompts_auto, mascara_gt, vertebras_eval=None):
    if vertebras_eval is None:
        vertebras_eval = vertebras_gt_eval(mascara_gt)

    filas = []
    for vertebra in vertebras_eval:
        gt = mask_binaria_vertebra(mascara_gt, vertebra).astype(bool)
        total_gt = int(gt.sum())
        if total_gt == 0:
            continue

        mejor = mejor_prompt_para_gt(prompts_auto, gt)
        if vertebra in prompts_auto:
            met_exacta = metricas_bbox_contra_gt(gt, prompts_auto[vertebra]["bbox_xyxy"])
            estado_prompt = "evaluado_gt"
            confianza_nn = prompts_auto[vertebra].get("confianza_nn", np.nan)
            presencia_nn = prompts_auto[vertebra].get("presencia_nn", np.nan)
        else:
            met_exacta = {
                "bbox_iou": 0.0,
                "bbox_recall": 0.0,
                "bbox_precision": 0.0,
                "center_error_px": np.nan,
            }
            estado_prompt = "gt_sin_prompt"
            confianza_nn = np.nan
            presencia_nn = np.nan

        fila = {
            "vertebra": vertebra,
            "id_real": VERTEBRA_TO_ID[vertebra],
            "bbox_iou": met_exacta["bbox_iou"],
            "bbox_recall": met_exacta["bbox_recall"],
            "bbox_precision": met_exacta["bbox_precision"],
            "center_error_px": met_exacta["center_error_px"],
            "confianza_nn": confianza_nn,
            "presencia_nn": presencia_nn,
            "estado_prompt": estado_prompt,
        }

        if mejor is not None:
            fila.update(mejor)
            fila["flexible_misma_etiqueta"] = bool(mejor["prompt_flexible_vertebra"] == vertebra)
            fila["desfase_id_flexible"] = int(mejor["prompt_flexible_id"] - VERTEBRA_TO_ID[vertebra])
            fila["mejora_iou_flexible"] = float(mejor["bbox_iou_flexible"] - met_exacta["bbox_iou"])
        else:
            fila.update({
                "prompt_flexible_vertebra": "",
                "prompt_flexible_id": np.nan,
                "bbox_iou_flexible": np.nan,
                "bbox_recall_flexible": np.nan,
                "bbox_precision_flexible": np.nan,
                "center_error_flexible_px": np.nan,
                "confianza_flexible": np.nan,
                "flexible_misma_etiqueta": False,
                "desfase_id_flexible": np.nan,
                "mejora_iou_flexible": np.nan,
            })

        filas.append(fila)

    if not filas:
        return pd.DataFrame(columns=[
            "vertebra", "id_real", "bbox_iou", "bbox_recall", "bbox_precision",
            "center_error_px", "confianza_nn", "presencia_nn", "estado_prompt",
            "prompt_flexible_vertebra", "prompt_flexible_id", "bbox_iou_flexible",
            "bbox_recall_flexible", "bbox_precision_flexible", "center_error_flexible_px",
            "flexible_misma_etiqueta", "desfase_id_flexible", "mejora_iou_flexible",
        ])

    return pd.DataFrame(filas).sort_values("id_real").reset_index(drop=True)


def resumen_etiquetas_prompts(prompts_auto, vertebras_gt):
    labels_prompts = sorted([v for v in prompts_auto.keys() if v in VERTEBRA_TO_ID], key=lambda v: VERTEBRA_TO_ID[v])
    labels_gt = list(vertebras_gt)
    labels_eval = [v for v in labels_prompts if v in labels_gt]
    labels_extra = [v for v in labels_prompts if v not in labels_gt]
    labels_faltantes = [v for v in labels_gt if v not in labels_prompts]
    return labels_prompts, labels_eval, labels_extra, labels_faltantes


def describir_desfases_flexibles(df, min_mejora=0.05):
    if df.empty or "prompt_flexible_vertebra" not in df:
        return ""
    filas = []
    for _, row in df.iterrows():
        prompt = row.get("prompt_flexible_vertebra", "")
        if not prompt or prompt == row["vertebra"]:
            continue
        mejora = float(row.get("mejora_iou_flexible", 0.0))
        iou_flex = float(row.get("bbox_iou_flexible", 0.0))
        if mejora >= min_mejora:
            filas.append(f"{row['vertebra']}<-{prompt}({iou_flex:.2f})")
    return ";".join(filas)


def evaluar_muestra_nn(split, patient_id, modo_etiquetado=None):
    t0 = time.perf_counter()
    img, mask, prompts, df_centros, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=modo_etiquetado)
    vertebras_gt = vertebras_gt_eval(mask)
    df = evaluar_cajas(prompts, mask, vertebras_eval=vertebras_gt)
    labels_prompts, labels_eval, labels_extra, labels_faltantes = resumen_etiquetas_prompts(prompts, vertebras_gt)
    tipo_real = "escoliosis" if str(patient_id).startswith("S_") else "normal"

    return {
        "split": split,
        "patient_id": patient_id,
        "tipo_real": tipo_real,
        "modo_evaluacion": "estricta_y_flexible_solo_etiquetas_gt",
        "estrategia_etiquetado": df_centros["estrategia_etiquetado"].iloc[0] if not df_centros.empty else (modo_etiquetado or ETIQUETADO_MODO),
        "estrategia_resuelta": df_centros["estrategia_resuelta"].iloc[0] if not df_centros.empty else "",
        "n_camino": int(df_centros["n_camino"].iloc[0]) if not df_centros.empty and "n_camino" in df_centros else 0,
        "n_gt": int(len(vertebras_gt)),
        "n_prompts": int(len(prompts)),
        "n_prompts_evaluados": int(len(labels_eval)),
        "n_prompts_extra_no_evaluados": int(len(labels_extra)),
        "n_gt_sin_prompt": int(len(labels_faltantes)),
        "labels_gt": ",".join(vertebras_gt),
        "labels_prompts_evaluados": ",".join(labels_eval),
        "labels_prompts_extra_no_evaluados": ",".join(labels_extra),
        "labels_gt_sin_prompt": ",".join(labels_faltantes),
        "tiempo_s": time.perf_counter() - t0,
        "bbox_iou_promedio": float(df["bbox_iou"].mean()) if not df.empty else np.nan,
        "bbox_recall_promedio": float(df["bbox_recall"].mean()) if not df.empty else np.nan,
        "bbox_precision_promedio": float(df["bbox_precision"].mean()) if not df.empty else np.nan,
        "n_vertebras_iou_mayor_02": int((df["bbox_iou"] > 0.20).sum()) if not df.empty else 0,
        "center_error_px": float(df["center_error_px"].mean()) if not df.empty else np.nan,
        "bbox_iou_flexible_promedio": float(df["bbox_iou_flexible"].mean()) if not df.empty else np.nan,
        "bbox_recall_flexible_promedio": float(df["bbox_recall_flexible"].mean()) if not df.empty else np.nan,
        "bbox_precision_flexible_promedio": float(df["bbox_precision_flexible"].mean()) if not df.empty else np.nan,
        "n_vertebras_flexible_iou_mayor_02": int((df["bbox_iou_flexible"] > 0.20).sum()) if not df.empty else 0,
        "center_error_flexible_px": float(df["center_error_flexible_px"].mean()) if not df.empty else np.nan,
        "n_flexible_misma_etiqueta": int(df["flexible_misma_etiqueta"].sum()) if not df.empty else 0,
        "mejora_iou_flexible_promedio": float(df["mejora_iou_flexible"].mean()) if not df.empty else np.nan,
        "desfase_id_flexible_promedio": float(df["desfase_id_flexible"].mean()) if not df.empty else np.nan,
        "desfases_flexibles": describir_desfases_flexibles(df),
        "confianza_media": float(df["confianza_nn"].mean()) if "confianza_nn" in df else np.nan,
    }, df


def evaluar_split_nn(split="val", patient_ids=None, modo_etiquetado=None):
    if patient_ids is None:
        patient_ids = sorted(PROMPTS_DICC[split].keys())

    resumen, detalles, errores = [], [], []
    for pid in tqdm(patient_ids, desc=f"Evaluando {split}"):
        try:
            row, df = evaluar_muestra_nn(split, pid, modo_etiquetado=modo_etiquetado)
            resumen.append(row)
            detalles.append(df.assign(split=split, patient_id=pid))
        except Exception as exc:
            errores.append({"split": split, "patient_id": pid, "error": repr(exc)})

    df_resumen = pd.DataFrame(resumen)
    df_detalle = pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame()
    df_errores = pd.DataFrame(errores)

    df_resumen.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_resumen.csv", index=False)
    df_detalle.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_detalle.csv", index=False)
    df_errores.to_csv(RESULTADOS_DIR / f"cajas_nn_centernet_val_errores.csv", index=False)
    return df_resumen, df_detalle, df_errores



def comparar_estrategias_etiquetado(split="val", patient_ids=None, estrategias=None):
    """Diagnostico opcional. La ruta principal usa solo top_anchor."""
    estrategias = estrategias or ESTRATEGIAS_ETIQUETADO
    frames = []
    for estrategia in estrategias:
        df_res, _, _ = evaluar_split_nn(split, patient_ids=patient_ids, modo_etiquetado=estrategia)
        if not df_res.empty:
            frames.append(df_res)

    df_cmp = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    out_path = RESULTADOS_DIR / f"cajas_nn_centernet_{split}_comparativo_estrategias.csv"
    df_cmp.to_csv(out_path, index=False)
    print("Comparativo guardado:", out_path)
    return df_cmp


## 9. Evaluacion VertebraPrompt-Net con correccion de shift

Aqui se evalua VertebraPrompt-Net en pacientes concretos y tambien se calcula el mejor `shift_anatomico`. Si una caja llamada `T6` cae mejor sobre `T4`, la metrica flexible lo detecta y el shift intenta explicar si el error es uniforme.

Esto fue necesario porque varias radiografias mostraban una columna bien recorrida pero etiquetas desplazadas. En esos casos la metrica estricta castiga fuerte, aunque visualmente el prompt pueda servir para MedSAM.

La conclusion de esta etapa fue que VertebraPrompt-Net es util como generador de prompts, pero todavia no resuelve por completo la asignacion anatomica en escoliosis.

<!-- codex-explicacion -->
El shift se conserva como diagnostico de desfase anatomico. Ayuda a distinguir entre mala caja y mala asignacion de nombre.


In [ ]:
# Casos trazadores usados para revisar visualmente normales y escoliosis.
PACIENTES_PRUEBA = ["N_12", "N_32", "S_187", "S_130", "S_80", "S_190"]
SHIFTS_ANATOMICOS = list(range(-6, 7))
IOU_PROMPT_UTIL = 0.20
RECALL_PROMPT_UTIL = 0.70


# Corrige un desfase anatomico uniforme sin mover las cajas.
def relabel_prompts_por_shift(prompts_auto, shift):
    """Desplaza nombres: si shift=+1, una caja T2 pasa a evaluarse como T1."""
    corregidos = {}
    asignaciones = []
    for old_label, info in prompts_auto.items():
        old_id = VERTEBRA_TO_ID.get(old_label)
        if old_id is None:
            continue
        new_id = old_id - int(shift)
        if new_id < 1 or new_id > N_CLASES:
            continue
        new_label = ID_TO_VERTEBRA[new_id]
        nuevo = dict(info)
        nuevo["vertebra_original"] = old_label
        nuevo["vertebra"] = new_label
        nuevo["id_real_original"] = old_id
        nuevo["id_real"] = new_id
        nuevo["shift_anatomico"] = int(shift)
        nuevo["prompt_origen"] = "vertebraprompt_net_shift"
        if new_label not in corregidos or nuevo.get("confianza_nn", 0) > corregidos[new_label].get("confianza_nn", 0):
            corregidos[new_label] = nuevo
        asignaciones.append({"vertebra_original": old_label, "vertebra_corregida": new_label, "shift": int(shift)})
    return corregidos, pd.DataFrame(asignaciones)


# Prueba varios shifts y escoge el que mejora recall/IoU contra GT disponible.
def evaluar_shifts_anatomicos(prompts_auto, mascara_gt, shifts=SHIFTS_ANATOMICOS):
    filas = []
    vertebras_gt = vertebras_gt_eval(mascara_gt)
    for shift in shifts:
        prompts_shift, _ = relabel_prompts_por_shift(prompts_auto, shift)
        df_shift = evaluar_cajas(prompts_shift, mascara_gt, vertebras_eval=vertebras_gt)
        filas.append({
            "shift": int(shift),
            "n_prompts_shift": int(len(prompts_shift)),
            "bbox_iou_shift": float(df_shift["bbox_iou"].mean()) if not df_shift.empty else np.nan,
            "bbox_recall_shift": float(df_shift["bbox_recall"].mean()) if not df_shift.empty else np.nan,
            "center_error_shift_px": float(df_shift["center_error_px"].mean()) if not df_shift.empty else np.nan,
            "n_vertebras_iou_mayor_02_shift": int((df_shift["bbox_iou"] > IOU_PROMPT_UTIL).sum()) if not df_shift.empty else 0,
            "n_vertebras_recall_mayor_07_shift": int((df_shift["bbox_recall"] > RECALL_PROMPT_UTIL).sum()) if not df_shift.empty else 0,
        })

    df_shifts = pd.DataFrame(filas)
    if df_shifts.empty:
        return 0, df_shifts, pd.DataFrame()

    orden = df_shifts.sort_values(
        ["bbox_recall_shift", "bbox_iou_shift", "n_vertebras_iou_mayor_02_shift", "shift"],
        ascending=[False, False, False, True],
    )
    best_shift = int(orden.iloc[0]["shift"])
    prompts_best, df_asignaciones = relabel_prompts_por_shift(prompts_auto, best_shift)
    df_best = evaluar_cajas(prompts_best, mascara_gt, vertebras_eval=vertebras_gt)
    return best_shift, df_shifts, df_best


def evaluar_muestra_nn_sam(split, patient_id, modo_etiquetado=None):
    t0 = time.perf_counter()
    img, mask, prompts, df_centros, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=modo_etiquetado)
    vertebras_gt = vertebras_gt_eval(mask)
    df_base = evaluar_cajas(prompts, mask, vertebras_eval=vertebras_gt)
    best_shift, df_shifts, df_shift = evaluar_shifts_anatomicos(prompts, mask)
    labels_prompts, labels_eval, labels_extra, labels_faltantes = resumen_etiquetas_prompts(prompts, vertebras_gt)
    tipo_real = "escoliosis" if str(patient_id).startswith("S_") else "normal"

    mejor_shift_row = df_shifts[df_shifts["shift"] == best_shift].iloc[0].to_dict() if not df_shifts.empty else {}
    return {
        "split": split,
        "patient_id": patient_id,
        "tipo_real": tipo_real,
        "estrategia_etiquetado": df_centros["estrategia_etiquetado"].iloc[0] if not df_centros.empty else (modo_etiquetado or ETIQUETADO_MODO),
        "estrategia_resuelta": df_centros["estrategia_resuelta"].iloc[0] if not df_centros.empty else "",
        "n_gt": int(len(vertebras_gt)),
        "n_prompts": int(len(prompts)),
        "n_prompts_extra_no_evaluados": int(len(labels_extra)),
        "labels_gt": ",".join(vertebras_gt),
        "labels_prompts_extra_no_evaluados": ",".join(labels_extra),
        "tiempo_cajas_s": time.perf_counter() - t0,
        "bbox_iou_estricto": float(df_base["bbox_iou"].mean()) if not df_base.empty else np.nan,
        "bbox_recall_estricto": float(df_base["bbox_recall"].mean()) if not df_base.empty else np.nan,
        "center_error_estricto_px": float(df_base["center_error_px"].mean()) if not df_base.empty else np.nan,
        "bbox_iou_flexible": float(df_base["bbox_iou_flexible"].mean()) if not df_base.empty else np.nan,
        "bbox_recall_flexible": float(df_base["bbox_recall_flexible"].mean()) if not df_base.empty else np.nan,
        "center_error_flexible_px": float(df_base["center_error_flexible_px"].mean()) if not df_base.empty else np.nan,
        "n_prompt_util_iou_flexible": int((df_base["bbox_iou_flexible"] > IOU_PROMPT_UTIL).sum()) if not df_base.empty else 0,
        "n_prompt_util_recall_flexible": int((df_base["bbox_recall_flexible"] > RECALL_PROMPT_UTIL).sum()) if not df_base.empty else 0,
        "best_shift": best_shift,
        "bbox_iou_shift": float(mejor_shift_row.get("bbox_iou_shift", np.nan)),
        "bbox_recall_shift": float(mejor_shift_row.get("bbox_recall_shift", np.nan)),
        "center_error_shift_px": float(mejor_shift_row.get("center_error_shift_px", np.nan)),
        "n_vertebras_iou_mayor_02_shift": int(mejor_shift_row.get("n_vertebras_iou_mayor_02_shift", 0)),
        "n_vertebras_recall_mayor_07_shift": int(mejor_shift_row.get("n_vertebras_recall_mayor_07_shift", 0)),
        "desfases_flexibles": describir_desfases_flexibles(df_base),
    }, df_base, df_shifts, df_shift


def evaluar_split_nn_sam(split="val", patient_ids=None):
    if patient_ids is None:
        patient_ids = sorted(PROMPTS_DICC[split].keys())

    resumen, detalles_base, detalles_shift, curvas_shift, errores = [], [], [], [], []
    for pid in tqdm(patient_ids, desc=f"NN-SAM {split}"):
        try:
            row, df_base, df_shifts, df_shift = evaluar_muestra_nn_sam(split, pid, modo_etiquetado=ETIQUETADO_MODO)
            resumen.append(row)
            detalles_base.append(df_base.assign(split=split, patient_id=pid))
            detalles_shift.append(df_shift.assign(split=split, patient_id=pid, best_shift=row["best_shift"]))
            curvas_shift.append(df_shifts.assign(split=split, patient_id=pid))
        except Exception as exc:
            errores.append({"split": split, "patient_id": pid, "error": repr(exc)})

    df_resumen = pd.DataFrame(resumen)
    df_base_all = pd.concat(detalles_base, ignore_index=True) if detalles_base else pd.DataFrame()
    df_shift_all = pd.concat(detalles_shift, ignore_index=True) if detalles_shift else pd.DataFrame()
    df_shifts_all = pd.concat(curvas_shift, ignore_index=True) if curvas_shift else pd.DataFrame()
    df_errores = pd.DataFrame(errores)

    df_resumen.to_csv(NN_SAM_DIR / f"nn_sam_{split}_resumen.csv", index=False)
    df_base_all.to_csv(NN_SAM_DIR / f"nn_sam_{split}_detalle_base.csv", index=False)
    df_shift_all.to_csv(NN_SAM_DIR / f"nn_sam_{split}_detalle_shift.csv", index=False)
    df_shifts_all.to_csv(NN_SAM_DIR / f"nn_sam_{split}_curvas_shift.csv", index=False)
    df_errores.to_csv(NN_SAM_DIR / f"nn_sam_{split}_errores.csv", index=False)
    return df_resumen, df_base_all, df_shift_all, df_shifts_all, df_errores


## 10. Evaluacion de prompts en validacion y test

Primero se mide la red sin MedSAM. Esta etapa responde si las prompts automaticos son suficientemente buenas para usarlas como prompts.

La lectura de cajas en `test` fue consistente con `val`: la metrica flexible quedo claramente por encima de la estricta. Esto indica que la red localiza vertebras, pero algunas quedan nombradas con desfase. Esa diferencia justifica pasar a MedSAM y evaluar segmentacion flexible ademas de estricta.

<!-- codex-explicacion -->
Primero se mide la calidad del prompt antes de culpar a MedSAM. Si el prompt falla, la mascara final casi siempre se degrada.


In [ ]:
patient_ids_eval = sorted(PROMPTS_DICC["val"].keys()) if EVALUAR_TODO_VAL else [p for p in PACIENTES_PRUEBA if p in PROMPTS_DICC["val"]]
df_nn_sam_resumen, df_nn_sam_detalle_base, df_nn_sam_detalle_shift, df_nn_sam_curvas_shift, df_nn_sam_errores = evaluar_split_nn_sam("val", patient_ids_eval)

display(df_nn_sam_resumen)

if not df_nn_sam_resumen.empty:
    display(
        df_nn_sam_resumen.groupby("tipo_real", as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            tiempo_cajas_s=("tiempo_cajas_s", "mean"),
            iou_estricto=("bbox_iou_estricto", "mean"),
            recall_estricto=("bbox_recall_estricto", "mean"),
            iou_flexible=("bbox_iou_flexible", "mean"),
            recall_flexible=("bbox_recall_flexible", "mean"),
            iou_shift=("bbox_iou_shift", "mean"),
            recall_shift=("bbox_recall_shift", "mean"),
            prompt_util_iou=("n_prompt_util_iou_flexible", "mean"),
            prompt_util_recall=("n_prompt_util_recall_flexible", "mean"),
        )
    )

display(df_nn_sam_errores)

if EVALUAR_TEST_FINAL and "test" in PROMPTS_DICC and len(PROMPTS_DICC["test"]) > 0:
    patient_ids_test = sorted(PROMPTS_DICC["test"].keys())
    df_nn_sam_test_resumen, df_nn_sam_test_detalle_base, df_nn_sam_test_detalle_shift, df_nn_sam_test_curvas_shift, df_nn_sam_test_errores = evaluar_split_nn_sam("test", patient_ids_test)
    display(df_nn_sam_test_resumen)
    display(df_nn_sam_test_errores)


## 11. Visualizacion de prompts estrictos, flexibles y shift

La visualizacion permite revisar si los numeros coinciden con la imagen. Se muestran tres paneles:

- Estricta: cajas comparadas contra la misma etiqueta.
- Flexible: mejor caja para cada GT, aunque tenga otro nombre.
- Shift: correccion anatomica uniforme estimada.

Esta parte fue importante porque casos como `S_80` pueden parecer malos en metrica estricta, pero visualmente muestran una columna segmentada de forma util. Sin imagenes, se podria concluir equivocadamente que la deteccion fallo por completo.

<!-- codex-explicacion -->
La visualizacion permite ver si las cajas van por el centro de las vertebras, si se repiten o si se desplazan. Fue una parte clave de la decision final.


In [ ]:
def dibujar_bbox(ax, bbox, label, color, linewidth=1.2):
    x0, y0, x1, y1 = bbox
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=color, linewidth=linewidth))
    ax.text(x0, max(0, y0 - 3), label, fontsize=7, color="white", bbox=dict(facecolor="black", alpha=0.45, pad=1))


def visualizar_nn_sam(split, patient_id, guardar=True):
    img, mask, prompts, df_centros, df_candidatos, out = generar_prompts_nn(split, patient_id, modo_etiquetado=ETIQUETADO_MODO)
    vertebras_gt = vertebras_gt_eval(mask)
    df_base = evaluar_cajas(prompts, mask, vertebras_eval=vertebras_gt)
    best_shift, df_shifts, df_shift = evaluar_shifts_anatomicos(prompts, mask)
    prompts_shift, df_asig = relabel_prompts_por_shift(prompts, best_shift)

    tab_base = df_base.set_index("vertebra") if not df_base.empty else pd.DataFrame()
    tab_shift = df_shift.set_index("vertebra") if not df_shift.empty else pd.DataFrame()
    overlay = np.zeros_like(img)
    overlay[..., 0] = (mask > 0).astype(np.uint8) * 255

    fig, ax = plt.subplots(1, 3, figsize=(22, 9))
    titulos = [
        "Estricta: misma etiqueta",
        "Flexible: mejor caja para cada GT",
        f"Shift anatomico: {best_shift:+d}",
    ]
    for a, titulo in zip(ax, titulos):
        a.imshow(img)
        a.imshow(overlay, alpha=0.16)
        a.set_title(titulo)
        a.axis("off")

    for vertebra, info in prompts.items():
        if vertebra not in vertebras_gt:
            continue
        iou = float(tab_base.loc[vertebra, "bbox_iou"]) if vertebra in tab_base.index else 0.0
        dibujar_bbox(ax[0], info["bbox_xyxy"], f"{vertebra} {iou:.2f}", "lime")

    for _, row in df_base.iterrows():
        gt_v = row["vertebra"]
        best_v = row.get("prompt_flexible_vertebra", "")
        if not best_v or best_v not in prompts:
            continue
        color = "lime" if best_v == gt_v else "gold"
        dibujar_bbox(ax[1], prompts[best_v]["bbox_xyxy"], f"GT {gt_v}<-{best_v} {float(row['bbox_iou_flexible']):.2f}", color)

    for vertebra, info in prompts_shift.items():
        if vertebra not in vertebras_gt:
            continue
        iou = float(tab_shift.loc[vertebra, "bbox_iou"]) if vertebra in tab_shift.index else 0.0
        old = info.get("vertebra_original", vertebra)
        dibujar_bbox(ax[2], info["bbox_xyxy"], f"{vertebra}<-{old} {iou:.2f}", "cyan")

    iou_gt = float(df_base["bbox_iou"].mean()) if not df_base.empty else np.nan
    iou_fx = float(df_base["bbox_iou_flexible"].mean()) if not df_base.empty else np.nan
    iou_sh = float(df_shift["bbox_iou"].mean()) if not df_shift.empty else np.nan
    recall_fx = float(df_base["bbox_recall_flexible"].mean()) if not df_base.empty else np.nan
    recall_sh = float(df_shift["bbox_recall"].mean()) if not df_shift.empty else np.nan
    fig.suptitle(
        f"{patient_id} | IoU estricto={iou_gt:.3f} | IoU flex={iou_fx:.3f} | IoU shift={iou_sh:.3f} "
        f"| recall flex={recall_fx:.3f} | recall shift={recall_sh:.3f} | GT={len(vertebras_gt)}",
        fontsize=13,
    )
    plt.tight_layout()

    if guardar:
        out_path = NN_SAM_DIR / f"visual_nn_sam_{split}_{patient_id}_shift_{best_shift:+d}.png"
        plt.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    display(df_centros)
    display(df_shifts.sort_values(["bbox_recall_shift", "bbox_iou_shift"], ascending=False).head(8))
    display(pd.DataFrame([{
        "patient_id": patient_id,
        "best_shift": best_shift,
        "labels_gt": ",".join(vertebras_gt),
        "desfases_flexibles": describir_desfases_flexibles(df_base),
    }]))
    display(df_base)
    display(df_shift)
    return prompts, prompts_shift, df_base, df_shift, df_shifts


for pid in [p for p in PACIENTES_VISUALIZACION if p in PROMPTS_DICC["val"]]:
    visualizar_nn_sam("val", pid)


## 12. BoxRefiner local

El detector ganador ya ubica la columna de forma razonable, pero algunas cajas quedan corridas o con tamano imperfecto. Esta seccion entrena un refinador local: toma un crop alrededor de la caja predicha y predice una correccion `dx, dy, dw, dh` hacia la caja real de la vertebra mas compatible.

La identidad anatomica no se cambia aqui; el objetivo es mejorar la caja que recibe MedSAM.

<!-- codex-explicacion -->
BoxRefiner ajusta cada caja usando informacion local. La motivacion fue corregir pequenos desplazamientos que afectaban MedSAM sin redisenar todo el pipeline.


In [ ]:
def centro_bbox_xyxy(bbox):
    x0, y0, x1, y1 = [float(v) for v in bbox]
    return (x0 + x1) / 2, (y0 + y1) / 2


def bbox_wh_xyxy(bbox):
    x0, y0, x1, y1 = [float(v) for v in bbox]
    return max(x1 - x0, 1.0), max(y1 - y0, 1.0)


def bbox_gt_desde_mask(mask_bin):
    ys, xs = np.where(mask_bin)
    if len(xs) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]


def mejor_gt_bbox_para_prompt(mask_gt, bbox_pred):
    mejor = None
    mejor_key = None
    for vertebra in vertebras_gt_eval(mask_gt):
        gt_bin = mask_binaria_vertebra(mask_gt, vertebra).astype(bool)
        bbox_gt = bbox_gt_desde_mask(gt_bin)
        if bbox_gt is None:
            continue
        met = metricas_bbox_contra_gt(gt_bin, bbox_pred)
        center = met["center_error_px"]
        key = (met["bbox_iou"], met["bbox_recall"], -center if np.isfinite(center) else -1e9)
        if mejor is None or key > mejor_key:
            mejor = {"vertebra_gt": vertebra, "bbox_gt": bbox_gt, **met}
            mejor_key = key
    if mejor is None:
        return None
    if mejor["bbox_iou"] < BOX_REFINER_MIN_IOU_TARGET and mejor["bbox_recall"] < BOX_REFINER_MIN_RECALL_TARGET:
        return None
    return mejor


def delta_box_target(bbox_pred, bbox_gt):
    cx, cy = centro_bbox_xyxy(bbox_pred)
    gx, gy = centro_bbox_xyxy(bbox_gt)
    bw, bh = bbox_wh_xyxy(bbox_pred)
    gw, gh = bbox_wh_xyxy(bbox_gt)
    return np.array([
        np.clip((gx - cx) / bw, -BOX_REFINER_MAX_ABS_DXY, BOX_REFINER_MAX_ABS_DXY),
        np.clip((gy - cy) / bh, -BOX_REFINER_MAX_ABS_DXY, BOX_REFINER_MAX_ABS_DXY),
        np.clip(np.log(gw / bw), -BOX_REFINER_MAX_ABS_LOG_SCALE, BOX_REFINER_MAX_ABS_LOG_SCALE),
        np.clip(np.log(gh / bh), -BOX_REFINER_MAX_ABS_LOG_SCALE, BOX_REFINER_MAX_ABS_LOG_SCALE),
    ], dtype=np.float32)


def aplicar_delta_bbox(bbox_pred, delta, img_shape, blend=BOX_REFINER_BLEND):
    H, W = img_shape[:2]
    dx, dy, dw, dh = [float(v) for v in delta]
    dx = np.clip(dx, -BOX_REFINER_MAX_ABS_DXY, BOX_REFINER_MAX_ABS_DXY) * blend
    dy = np.clip(dy, -BOX_REFINER_MAX_ABS_DXY, BOX_REFINER_MAX_ABS_DXY) * blend
    dw = np.clip(dw, -BOX_REFINER_MAX_ABS_LOG_SCALE, BOX_REFINER_MAX_ABS_LOG_SCALE) * blend
    dh = np.clip(dh, -BOX_REFINER_MAX_ABS_LOG_SCALE, BOX_REFINER_MAX_ABS_LOG_SCALE) * blend
    cx, cy = centro_bbox_xyxy(bbox_pred)
    bw, bh = bbox_wh_xyxy(bbox_pred)
    cx2 = cx + dx * bw
    cy2 = cy + dy * bh
    bw2 = bw * np.exp(dw)
    bh2 = bh * np.exp(dh)
    return bbox_clip([cx2 - bw2 / 2, cy2 - bh2 / 2, cx2 + bw2 / 2, cy2 + bh2 / 2], H, W)


def expandir_bbox_contexto(bbox, img_shape, frac=BOX_REFINER_CONTEXT_FRAC):
    H, W = img_shape[:2]
    x0, y0, x1, y1 = [float(v) for v in bbox]
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    bw, bh = max(x1 - x0, 2.0), max(y1 - y0, 2.0)
    side = max(bw, bh) * (1.0 + float(frac))
    return bbox_clip([cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2], H, W)


def box_refiner_tensor(img_rgb, bbox_pred, augment=False):
    ctx = expandir_bbox_contexto(bbox_pred, img_rgb.shape)
    x0, y0, x1, y1 = ctx
    if augment:
        H, W = img_rgb.shape[:2]
        cw, ch = x1 - x0, y1 - y0
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        cx += np.random.uniform(-0.06, 0.06) * cw
        cy += np.random.uniform(-0.06, 0.06) * ch
        side = max(cw, ch) * np.random.uniform(0.92, 1.12)
        ctx = bbox_clip([cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2], H, W)
        x0, y0, x1, y1 = ctx

    crop = Image.fromarray(img_rgb).convert("L").crop((int(x0), int(y0), int(x1) + 1, int(y1) + 1)).resize((BOX_REFINER_SIZE, BOX_REFINER_SIZE), Image.BILINEAR)
    arr = np.asarray(crop).astype(np.float32)
    p1, p99 = np.percentile(arr, [1, 99.5])
    arr = np.clip((arr - p1) / (p99 - p1 + 1e-6), 0, 1)
    if augment:
        arr = np.clip(arr * np.random.uniform(0.92, 1.08) + np.random.uniform(-0.04, 0.04), 0, 1)

    mask_box = np.zeros((BOX_REFINER_SIZE, BOX_REFINER_SIZE), dtype=np.float32)
    bx0, by0, bx1, by1 = [float(v) for v in bbox_pred]
    sx = BOX_REFINER_SIZE / max(x1 - x0 + 1, 1)
    sy = BOX_REFINER_SIZE / max(y1 - y0 + 1, 1)
    rx0 = int(np.clip(round((bx0 - x0) * sx), 0, BOX_REFINER_SIZE - 1))
    rx1 = int(np.clip(round((bx1 - x0) * sx), 0, BOX_REFINER_SIZE - 1))
    ry0 = int(np.clip(round((by0 - y0) * sy), 0, BOX_REFINER_SIZE - 1))
    ry1 = int(np.clip(round((by1 - y0) * sy), 0, BOX_REFINER_SIZE - 1))
    if rx1 > rx0 and ry1 > ry0:
        mask_box[ry0:ry1 + 1, rx0:rx1 + 1] = 1.0

    x = np.stack([arr, mask_box], axis=0).astype(np.float32)
    return torch.from_numpy(x)


def construir_dataset_box_refiner(split="train", patient_ids=None, regenerar=False):
    cache_path = RESULTADOS_DIR / f"box_refiner_{split}_samples.csv"
    if cache_path.exists() and not regenerar:
        return pd.read_csv(cache_path)
    patient_ids = patient_ids or sorted(PROMPTS_DICC[split].keys())
    filas, errores = [], []
    for patient_id in tqdm(patient_ids, desc=f"box-refiner samples {split}"):
        try:
            img, mask_gt, prompts_auto, _, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=ETIQUETADO_MODO)
            for raw_label, info in prompts_auto.items():
                bbox_pred = [int(v) for v in info["bbox_xyxy"]]
                match = mejor_gt_bbox_para_prompt(mask_gt, bbox_pred)
                if match is None:
                    continue
                delta = delta_box_target(bbox_pred, match["bbox_gt"])
                filas.append({
                    "split": split,
                    "patient_id": patient_id,
                    "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
                    "raw_label": raw_label,
                    "vertebra_gt_match": match["vertebra_gt"],
                    "bbox_pred": json.dumps(bbox_pred),
                    "bbox_gt": json.dumps(match["bbox_gt"]),
                    "bbox_iou_match": float(match["bbox_iou"]),
                    "bbox_recall_match": float(match["bbox_recall"]),
                    "dx": float(delta[0]),
                    "dy": float(delta[1]),
                    "dw": float(delta[2]),
                    "dh": float(delta[3]),
                })
        except Exception as exc:
            errores.append({"split": split, "patient_id": patient_id, "error": repr(exc)})
    df = pd.DataFrame(filas)
    df.to_csv(cache_path, index=False)
    if errores:
        pd.DataFrame(errores).to_csv(RESULTADOS_DIR / f"box_refiner_{split}_errores.csv", index=False)
    print(f"BoxRefiner {split}: {len(df)} muestras | {cache_path}")
    return df


class BoxRefinerDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path_img, _ = resolver_paths_muestra(row["split"], row["patient_id"])
        img = cargar_imagen(path_img)
        bbox_pred = json.loads(row["bbox_pred"])
        x = box_refiner_tensor(img, bbox_pred, augment=self.augment)
        y = torch.tensor([row["dx"], row["dy"], row["dw"], row["dh"]], dtype=torch.float32)
        return x, y


class BoxRefinerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(2, 24), nn.MaxPool2d(2),
            ConvBlock(24, 48), nn.MaxPool2d(2),
            ConvBlock(48, 96), nn.MaxPool2d(2),
            ConvBlock(96, 128),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.15),
            nn.Linear(128, 96),
            nn.SiLU(inplace=True),
            nn.Linear(96, 4),
        )

    def forward(self, x):
        out = self.net(x)
        dxy = torch.tanh(out[:, :2]) * BOX_REFINER_MAX_ABS_DXY
        dwh = torch.tanh(out[:, 2:]) * BOX_REFINER_MAX_ABS_LOG_SCALE
        return torch.cat([dxy, dwh], dim=1)


@torch.no_grad()
def evaluar_box_refiner_loader(model_refiner, loader):
    model_refiner.eval()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        pred = model_refiner(x)
        losses.append(float(F.smooth_l1_loss(pred, y).item()))
    return float(np.mean(losses)) if losses else np.nan


def entrenar_box_refiner():
    df_train = construir_dataset_box_refiner("train", regenerar=False)
    df_val = construir_dataset_box_refiner("val", regenerar=False)
    train_loader = DataLoader(BoxRefinerDataset(df_train, augment=True), batch_size=BOX_REFINER_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    val_loader = DataLoader(BoxRefinerDataset(df_val, augment=False), batch_size=BOX_REFINER_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

    model_refiner = BoxRefinerNet().to(DEVICE)
    opt = torch.optim.AdamW(model_refiner.parameters(), lr=BOX_REFINER_LR, weight_decay=BOX_REFINER_WEIGHT_DECAY)
    best_val, patience = np.inf, 0
    hist = []
    for epoch in range(1, BOX_REFINER_EPOCHS + 1):
        model_refiner.train()
        losses = []
        for x, y in tqdm(train_loader, desc=f"BoxRefiner epoch {epoch}", leave=False):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            pred = model_refiner(x)
            loss = F.smooth_l1_loss(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_refiner.parameters(), 1.0)
            opt.step()
            losses.append(float(loss.item()))
        val_loss = evaluar_box_refiner_loader(model_refiner, val_loader)
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "val_loss": val_loss, "train_n": len(df_train), "val_n": len(df_val)}
        hist.append(row)
        print(row)
        if val_loss < best_val - 1e-5:
            best_val = val_loss
            patience = 0
            torch.save({"model_state_dict": model_refiner.state_dict(), "best_epoch": epoch, "best_val_loss": best_val}, BOX_REFINER_CKPT_PATH)
        else:
            patience += 1
            if patience >= BOX_REFINER_PACIENCIA:
                print("Early stopping BoxRefiner")
                break
    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(RESULTADOS_DIR / "historial_box_refiner.csv", index=False)
    return cargar_box_refiner(), hist_df


def cargar_box_refiner():
    if not BOX_REFINER_CKPT_PATH.exists():
        raise FileNotFoundError(f"No existe checkpoint BoxRefiner: {BOX_REFINER_CKPT_PATH}")
    ckpt = torch_load_compatible(BOX_REFINER_CKPT_PATH, map_location=DEVICE) if "torch_load_compatible" in globals() else torch.load(BOX_REFINER_CKPT_PATH, map_location=DEVICE)
    model_refiner = BoxRefinerNet().to(DEVICE)
    model_refiner.load_state_dict(ckpt["model_state_dict"])
    model_refiner.eval()
    model_refiner.best_epoch = ckpt.get("best_epoch")
    model_refiner.best_val_loss = ckpt.get("best_val_loss")
    print("BoxRefiner cargado:", BOX_REFINER_CKPT_PATH, "best_epoch:", model_refiner.best_epoch)
    return model_refiner


@torch.no_grad()
def refinar_prompts_con_box_refiner(prompts_auto, img_rgb, model_refiner):
    refinados, filas = {}, []
    for label, info in prompts_auto.items():
        bbox_pred = [int(v) for v in info["bbox_xyxy"]]
        x = box_refiner_tensor(img_rgb, bbox_pred, augment=False)[None].to(DEVICE)
        delta = model_refiner(x)[0].detach().cpu().numpy()
        bbox_ref = aplicar_delta_bbox(bbox_pred, delta, img_rgb.shape)
        nuevo = dict(info)
        nuevo["bbox_xyxy_original"] = bbox_pred
        nuevo["bbox_xyxy"] = bbox_ref
        nuevo["box_refiner_delta"] = [float(v) for v in delta]
        nuevo["prompt_origen"] = "vertebraprompt_box_refiner"
        refinados[label] = nuevo
        filas.append({"vertebra": label, "bbox_original": bbox_pred, "bbox_refinado": bbox_ref, "dx": float(delta[0]), "dy": float(delta[1]), "dw": float(delta[2]), "dh": float(delta[3])})
    return refinados, pd.DataFrame(filas)


def evaluar_box_refiner_split(split="val", patient_ids=None):
    if not USAR_BOX_REFINER or globals().get("box_refiner_model_final") is None:
        print("BoxRefiner inactivo")
        return pd.DataFrame()
    patient_ids = patient_ids or sorted(PROMPTS_DICC[split].keys())
    filas, detalles = [], []
    for patient_id in tqdm(patient_ids, desc=f"eval BoxRefiner {split}"):
        img, mask_gt, prompts_raw, _, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=ETIQUETADO_MODO)
        prompts_ref, df_ref = refinar_prompts_con_box_refiner(prompts_raw, img, box_refiner_model_final)
        vertebras_gt = vertebras_gt_eval(mask_gt)
        for nombre, prompts in [("raw", prompts_raw), ("box_refiner", prompts_ref)]:
            df_met = evaluar_cajas(prompts, mask_gt, vertebras_eval=vertebras_gt)
            filas.append({
                "split": split,
                "patient_id": patient_id,
                "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
                "estrategia": nombre,
                "n_gt": len(vertebras_gt),
                "n_prompts": len(prompts),
                "bbox_iou_estricto": float(df_met["bbox_iou"].mean()) if not df_met.empty else np.nan,
                "bbox_recall_estricto": float(df_met["bbox_recall"].mean()) if not df_met.empty else np.nan,
                "bbox_iou_flexible": float(df_met["bbox_iou_flexible"].mean()) if not df_met.empty else np.nan,
                "bbox_recall_flexible": float(df_met["bbox_recall_flexible"].mean()) if not df_met.empty else np.nan,
                "center_error_flexible_px": float(df_met["center_error_flexible_px"].mean()) if not df_met.empty else np.nan,
            })
            if not df_met.empty:
                detalles.append(df_met.assign(split=split, patient_id=patient_id, estrategia=nombre))
    df_res = pd.DataFrame(filas)
    df_det = pd.concat(detalles, ignore_index=True) if detalles else pd.DataFrame()
    df_res.to_csv(RESULTADOS_DIR / f"box_refiner_{split}_resumen.csv", index=False)
    df_det.to_csv(RESULTADOS_DIR / f"box_refiner_{split}_detalle.csv", index=False)
    display(df_res.groupby(["estrategia", "tipo_real"], as_index=False).agg(
        pacientes=("patient_id", "nunique"),
        bbox_iou_estricto=("bbox_iou_estricto", "mean"),
        bbox_recall_estricto=("bbox_recall_estricto", "mean"),
        bbox_iou_flexible=("bbox_iou_flexible", "mean"),
        bbox_recall_flexible=("bbox_recall_flexible", "mean"),
        center_error_flexible_px=("center_error_flexible_px", "mean"),
    ))
    return df_res


if USAR_BOX_REFINER:
    if EJECUTAR_ENTRENAMIENTO_BOX_REFINER or not BOX_REFINER_CKPT_PATH.exists():
        box_refiner_model_final, hist_box_refiner = entrenar_box_refiner()
        display(hist_box_refiner.tail())
    else:
        box_refiner_model_final = cargar_box_refiner()
        hist_box_refiner = pd.DataFrame()
    df_box_refiner_val = evaluar_box_refiner_split("val", PACIENTES_VISUALIZACION)
else:
    box_refiner_model_final = None
    hist_box_refiner = pd.DataFrame()
    df_box_refiner_val = pd.DataFrame()

## 13. Exportar prompts VertebraPrompt-Net

Se exportan dos versiones de prompts:

- Base: cajas generadas por VertebraPrompt-Net sin cambiar nombres.
- Shift: cajas con el mejor corrimiento anatomico medido contra GT.

La version base representa mejor el pipeline automatico real. La version con shift sirve para diagnosticar cuanto se pierde por asignacion anatomica. En el flujo final, los prompts automaticos se usan como entrada de MedSAM.

<!-- codex-explicacion -->
Se exportan prompts refinados para que MedSAM pueda usarlos directamente. Esto hace reproducible la estrategia ganadora.


In [ ]:
# Exporta prompts para reproducir la prueba y alimentar MedSAM u otros modelos.
def exportar_prompts_nn_sam(split="val", patient_ids=None, usar_shift=False):
    if patient_ids is None:
        patient_ids = sorted(PROMPTS_DICC[split].keys())

    export = {}
    filas = []
    for pid in tqdm(patient_ids, desc=f"Export prompts NN-SAM {split}"):
        img, mask, prompts, df_centros, _, _ = generar_prompts_nn(split, pid, modo_etiquetado=ETIQUETADO_MODO)
        best_shift = 0
        prompts_out = prompts
        if usar_shift:
            best_shift, _, _ = evaluar_shifts_anatomicos(prompts, mask)
            prompts_out, _ = relabel_prompts_por_shift(prompts, best_shift)

        export[pid] = {}
        for vertebra, info in prompts_out.items():
            export[pid][vertebra] = {
                "bbox_xyxy": [int(v) for v in info["bbox_xyxy"]],
                "confianza_nn": float(info.get("confianza_nn", np.nan)),
                "presencia_nn": float(info.get("presencia_nn", np.nan)),
                "shift_anatomico": int(best_shift),
                "vertebra_original": info.get("vertebra_original", vertebra),
                "origen": info.get("prompt_origen", "cajas_nn_centernet_lite"),
            }
            filas.append({
                "split": split,
                "patient_id": pid,
                "vertebra": vertebra,
                "vertebra_original": info.get("vertebra_original", vertebra),
                "shift_anatomico": int(best_shift),
                "bbox_xyxy": export[pid][vertebra]["bbox_xyxy"],
                "confianza_nn": export[pid][vertebra]["confianza_nn"],
            })

    sufijo = "shift_gt" if usar_shift else "base"
    out_json = NN_SAM_DIR / f"prompts_nn_sam_{split}_{sufijo}.json"
    out_csv = NN_SAM_DIR / f"prompts_nn_sam_{split}_{sufijo}.csv"
    out_json.write_text(json.dumps(export, indent=2, ensure_ascii=False), encoding="utf-8")
    pd.DataFrame(filas).to_csv(out_csv, index=False)
    print("JSON:", out_json)
    print("CSV:", out_csv)
    return export, pd.DataFrame(filas)


prompts_nn_sam_base, df_prompts_nn_sam_base = exportar_prompts_nn_sam("val", patient_ids_eval, usar_shift=False)
prompts_nn_sam_shift, df_prompts_nn_sam_shift = exportar_prompts_nn_sam("val", patient_ids_eval, usar_shift=True)
display(df_prompts_nn_sam_shift.head())


if EVALUAR_TEST_FINAL and "test" in PROMPTS_DICC and len(PROMPTS_DICC["test"]) > 0:
    prompts_nn_sam_test_base, df_prompts_nn_sam_test_base = exportar_prompts_nn_sam("test", sorted(PROMPTS_DICC["test"].keys()), usar_shift=False)
    prompts_nn_sam_test_shift, df_prompts_nn_sam_test_shift = exportar_prompts_nn_sam("test", sorted(PROMPTS_DICC["test"].keys()), usar_shift=True)


## 14. MedSAM: entrenamiento y comparacion final

Despues de generar prompts automaticos, el siguiente paso fue entrenar MedSAM dentro del mismo notebook. Esto era importante para que el documento de entrega no dependiera de resultados producidos en otro archivo.

Se comparan tres escenarios:

- `medsam_general`: baseline sin fine-tuning local.
- `medsam_decoder`: se entrena solo el `mask_decoder`.
- `medsam_decoder_encoder_parcial`: se entrena el decoder y se libera el ultimo bloque del encoder.

La comparacion final usa prompts automaticos VertebraPrompt-Net, no cajas ideales derivadas del GT. Por eso mide mucho mejor el comportamiento real del pipeline.

<!-- codex-explicacion -->
Aqui se conecta el refinamiento de prompts con la segmentacion. La pregunta es si mejores cajas se traducen en mejor Dice/IoU.


## 15. Prompts para MedSAM

Aqui se define como se llama a MedSAM. La ruta ganadora fue `box_only`, que significa darle solamente la caja `[x0, y0, x1, y1]` como prompt.

Tambien existen modos diagnosticos con puntos positivos/negativos, pero no fueron la ruta principal. En estas pruebas, agregar puntos no era necesario para el objetivo final y podia introducir variabilidad adicional.

`sin_pad` significa que la caja se usa tal como la genero VertebraPrompt-Net, sin expansion adicional. En validacion, `sin_pad` gano frente a cajas expandidas, lo que sugiere que las cajas actuales ya tienen un tamano adecuado para MedSAM.

<!-- codex-explicacion -->
Se valida que los prompts tengan el formato correcto y correspondan a las vertebras esperadas antes de segmentar.


In [ ]:
# Expansion leve de caja antes de MedSAM; no cambia la deteccion, solo el prompt.
def expandir_bbox_xyxy_frac(bbox, img_shape, frac_x=0.0, frac_y=0.0):
    H, W = img_shape[:2]
    x0, y0, x1, y1 = [float(v) for v in bbox]
    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2
    bw = max(1.0, x1 - x0)
    bh = max(1.0, y1 - y0)
    dx = bw * float(frac_x)
    dy = bh * float(frac_y)
    return np.array([
        np.clip(x0 - dx, 0, W - 1),
        np.clip(y0 - dy, 0, H - 1),
        np.clip(x1 + dx, 0, W - 1),
        np.clip(y1 + dy, 0, H - 1),
    ], dtype=np.float32)


# Puntos opcionales de diagnostico; la ruta principal sigue siendo box_only.
def preparar_puntos_prompt(img_rgb, bbox, margen_neg=8):
    """Punto positivo interno y negativos alrededor. Queda como diagnostico, no como ruta principal."""
    H, W = img_rgb.shape[:2]
    x0, y0, x1, y1 = [int(v) for v in bbox]
    gray = np.array(Image.fromarray(img_rgb).convert("L"), dtype=np.float32)
    crop = gray[y0:y1 + 1, x0:x1 + 1]
    if crop.size == 0:
        pos = np.array([[(x0 + x1) / 2, (y0 + y1) / 2]], dtype=np.float32)
    else:
        thr = np.percentile(crop, 75)
        ys, xs = np.where(crop >= thr)
        if len(xs) == 0:
            pos = np.array([[(x0 + x1) / 2, (y0 + y1) / 2]], dtype=np.float32)
        else:
            weights = crop[ys, xs] + 1e-3
            px = x0 + float(np.average(xs, weights=weights))
            py = y0 + float(np.average(ys, weights=weights))
            pos = np.array([[px, py]], dtype=np.float32)

    neg = np.array([
        [max(0, x0 - margen_neg), max(0, y0 - margen_neg)],
        [min(W - 1, x1 + margen_neg), max(0, y0 - margen_neg)],
        [max(0, x0 - margen_neg), min(H - 1, y1 + margen_neg)],
        [min(W - 1, x1 + margen_neg), min(H - 1, y1 + margen_neg)],
    ], dtype=np.float32)
    coords = np.vstack([pos, neg])
    labels = np.array([1, 0, 0, 0, 0], dtype=np.int32)
    return coords, labels


# Construye los argumentos exactos para SamPredictor.predict.
def preparar_prompt_medsam(img_rgb, bbox, modo="box_only", bbox_frac_x=0.0, bbox_frac_y=0.0):
    """Construye los argumentos para SamPredictor.predict."""
    bbox_np = expandir_bbox_xyxy_frac(bbox, img_rgb.shape, bbox_frac_x, bbox_frac_y)
    if modo == "box_only":
        return {"box": bbox_np, "multimask_output": False}
    if modo == "box_center":
        x0, y0, x1, y1 = [float(v) for v in bbox_np]
        point_coords = np.array([[(x0 + x1) / 2, (y0 + y1) / 2]], dtype=np.float32)
        point_labels = np.array([1], dtype=np.int32)
        return {"box": bbox_np, "point_coords": point_coords, "point_labels": point_labels, "multimask_output": False}
    if modo == "box_borde":
        point_coords, point_labels = preparar_puntos_prompt(img_rgb, bbox_np)
        return {"box": bbox_np, "point_coords": point_coords, "point_labels": point_labels, "multimask_output": False}
    raise ValueError(f"Modo de prompt no soportado: {modo}")


def dice_iou_binario(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = int(np.logical_and(pred, gt).sum())
    union = int(np.logical_or(pred, gt).sum())
    denom = int(pred.sum() + gt.sum())
    return {
        "dice": (2 * inter / denom) if denom else np.nan,
        "iou": (inter / union) if union else np.nan,
        "pix_pred": int(pred.sum()),
        "pix_gt": int(gt.sum()),
    }


## 16. Entrenamiento de MedSAM dentro del notebook

MedSAM aprende a transformar una caja en una mascara vertebral. El entrenamiento se hace por instancia: una muestra corresponde a una imagen, una vertebra, una caja y una mascara binaria GT.

Se entrenan dos variantes:

1. Decoder: se congelan `image_encoder` y `prompt_encoder`; aprende solo `mask_decoder`.
2. Encoder parcial: se parte del decoder entrenado y se libera el ultimo bloque del `image_encoder`.

El forward del encoder parcial es diferenciable, para que el encoder realmente reciba gradiente. Esto fue clave: si se usara solo `SamPredictor.set_image()` para entrenar el encoder, podria no actualizarse correctamente.

La conclusion de entrenamiento fue positiva: MedSAM entrenado supero con claridad al baseline general. El encoder parcial dio una mejora adicional pequena pero consistente, por eso se tomo como mejor modelo.

<!-- codex-explicacion -->
MedSAM se ajusta con los prompts generados por la estrategia. Esto evita evaluar con cajas ideales y mantiene el pipeline automatico.


In [ ]:
from torch.utils.data import DataLoader


def collate_fn_medsam(batch):
    return batch


def extraer_state_dict_medsam_entrenamiento(state):
    """Acepta checkpoints guardados como state_dict directo o envueltos en claves comunes."""
    if isinstance(state, dict):
        for key in ["model", "model_state_dict", "state_dict", "medsam"]:
            if key in state and isinstance(state[key], dict):
                return state[key]
    return state


# Dataset por instancia vertebral: cada muestra es una imagen, una caja y la mascara binaria de una vertebra.
class MedSAMVertebraDatasetEntrega(Dataset):
    def __init__(self, split, frac_x=0.04, frac_y=0.06):
        self.split = split
        self.frac_x = frac_x
        self.frac_y = frac_y
        self.samples = []

        for patient_id, prompts_sample in PROMPTS_DICC[split].items():
            labels_gt = set(GT_LABELS_DICC.get(split, {}).get(patient_id, []))
            for vertebra, info in prompts_sample.items():
                if vertebra not in labels_gt:
                    continue
                self.samples.append({
                    "patient_id": patient_id,
                    "vertebra": vertebra,
                    "bbox": info["bbox_xyxy"],
                    "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
                })

        print(f"MedSAM {split}: {len(self.samples)} muestras vertebrales")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        path_img, path_mask = resolver_paths_muestra(self.split, item["patient_id"])
        img = cargar_imagen(path_img).astype(np.uint8)
        mask = cargar_mascara(path_mask)

        bbox = expandir_bbox_xyxy_frac(item["bbox"], img.shape, self.frac_x, self.frac_y)
        gt_bin = mask_binaria_vertebra(mask, item["vertebra"]).astype(np.float32)

        peso = MEDSAM_PESO_ESCOLIOSIS if item["tipo_real"] == "escoliosis" else MEDSAM_PESO_NORMAL
        if item["vertebra"] in MEDSAM_VERTEBRAS_DIFICILES:
            peso *= MEDSAM_PESO_VERTEBRA_DIFICIL

        return {
            "patient_id": item["patient_id"],
            "vertebra": item["vertebra"],
            "tipo_real": item["tipo_real"],
            "image": img,
            "box": torch.tensor(bbox, dtype=torch.float32),
            "gt_mask": torch.tensor(gt_bin[None, :, :], dtype=torch.float32),
            "sample_weight": torch.tensor(float(peso), dtype=torch.float32),
        }


# DataLoaders de MedSAM: batch pequeno por memoria GPU; collate conserva numpy images.
def crear_loaders_medsam_entrega():
    train_ds = MedSAMVertebraDatasetEntrega("train", frac_x=MEDSAM_TRAIN_FRAC_X, frac_y=MEDSAM_TRAIN_FRAC_Y)
    val_ds = MedSAMVertebraDatasetEntrega("val", frac_x=MEDSAM_TRAIN_FRAC_X, frac_y=MEDSAM_TRAIN_FRAC_Y)
    train_loader = DataLoader(
        train_ds,
        batch_size=MEDSAM_TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=MEDSAM_TRAIN_WORKERS,
        collate_fn=collate_fn_medsam,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=MEDSAM_TRAIN_BATCH_SIZE,
        shuffle=False,
        num_workers=MEDSAM_TRAIN_WORKERS,
        collate_fn=collate_fn_medsam,
    )
    return train_loader, val_loader


bce_loss_medsam = nn.BCEWithLogitsLoss()


def dice_loss_from_logits_medsam(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.reshape(probs.shape[0], -1)
    targets = targets.reshape(targets.shape[0], -1)
    inter = (probs * targets).sum(dim=1)
    denom = probs.sum(dim=1) + targets.sum(dim=1)
    dice = (2 * inter + eps) / (denom + eps)
    return 1 - dice.mean()


@torch.no_grad()
def dice_iou_from_logits_medsam(logits, targets):
    pred = (torch.sigmoid(logits) > 0.5).float()
    pred_f = pred.reshape(pred.shape[0], -1)
    tgt_f = targets.reshape(targets.shape[0], -1)
    inter = (pred_f * tgt_f).sum(dim=1)
    union = ((pred_f + tgt_f) > 0).float().sum(dim=1)
    denom = pred_f.sum(dim=1) + tgt_f.sum(dim=1)
    dice = ((2 * inter) / denom.clamp(min=1)).mean().item()
    iou = (inter / union.clamp(min=1)).mean().item()
    return dice, iou


# Carga MedSAM base; desde aqui parten decoder y encoder parcial.
def cargar_medsam_base_para_entrenar():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA no esta disponible. No entreno MedSAM en CPU.")
    if not MEDSAM_CKPT_PATH.exists():
        raise FileNotFoundError(f"No existe MedSAM base: {MEDSAM_CKPT_PATH}")
    from segment_anything import sam_model_registry
    medsam = sam_model_registry["vit_b"](checkpoint=str(MEDSAM_CKPT_PATH)).to(DEVICE)
    return medsam


# Congela/libera parametros segun la variante a entrenar.
def configurar_entrenabilidad_medsam(medsam, modo):
    for p in medsam.parameters():
        p.requires_grad = False

    for p in medsam.prompt_encoder.parameters():
        p.requires_grad = False

    for p in medsam.mask_decoder.parameters():
        p.requires_grad = True

    if modo == "decoder_encoder_parcial":
        if not hasattr(medsam.image_encoder, "blocks"):
            raise AttributeError("No encontre image_encoder.blocks para abrir el ultimo bloque del encoder.")
        for p in medsam.image_encoder.blocks[-1].parameters():
            p.requires_grad = True

    n_total = sum(p.numel() for p in medsam.parameters())
    n_train = sum(p.numel() for p in medsam.parameters() if p.requires_grad)
    print(f"Modo {modo}: parametros entrenables {n_train:,} / {n_total:,} ({100*n_train/n_total:.2f}%)")


# Usa un solo LR para decoder y dos LR cuando se libera encoder parcial.
def optimizador_medsam(medsam, modo):
    if modo == "decoder":
        return torch.optim.AdamW(
            [p for p in medsam.mask_decoder.parameters() if p.requires_grad],
            lr=MEDSAM_LR_DECODER,
            weight_decay=MEDSAM_WEIGHT_DECAY,
        )

    params_encoder = []
    params_decoder = []
    for name, p in medsam.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("image_encoder.blocks."):
            params_encoder.append(p)
        elif name.startswith("mask_decoder."):
            params_decoder.append(p)

    return torch.optim.AdamW(
        [
            {"params": params_encoder, "lr": MEDSAM_LR_ENCODER},
            {"params": params_decoder, "lr": MEDSAM_LR_DECODER_REFINO},
        ],
        weight_decay=MEDSAM_WEIGHT_DECAY,
    )


# Forward diferenciable: necesario para que el ultimo bloque del encoder pueda aprender.
def forward_medsam_diferenciable(medsam, image_np, box_tensor, entrenar_encoder=False):
    H, W = image_np.shape[:2]
    image_t = torch.as_tensor(image_np, dtype=torch.float32, device=DEVICE).permute(2, 0, 1).unsqueeze(0)
    input_image = medsam.preprocess(image_t)

    if entrenar_encoder:
        image_embedding = medsam.image_encoder(input_image)
    else:
        with torch.no_grad():
            image_embedding = medsam.image_encoder(input_image)

    box_torch = box_tensor.to(DEVICE).float().reshape(1, 4)
    with torch.no_grad():
        sparse_embeddings, dense_embeddings = medsam.prompt_encoder(points=None, boxes=box_torch, masks=None)

    low_res_masks, iou_predictions = medsam.mask_decoder(
        image_embeddings=image_embedding,
        image_pe=medsam.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=False,
    )

    masks = medsam.postprocess_masks(low_res_masks, input_size=(H, W), original_size=(H, W))
    return masks, iou_predictions


# Ejecuta una epoca de entrenamiento o validacion y reporta loss, Dice e IoU binarios.
def run_epoch_medsam_entrega(medsam, loader, optimizer=None, modo="decoder", epoch=None):
    train = optimizer is not None
    medsam.train(train)
    # Prompt encoder queda congelado aunque el modelo este en modo train.
    medsam.prompt_encoder.eval()
    if modo == "decoder":
        medsam.image_encoder.eval()
    desc = f"MedSAM {modo} epoch {epoch}" if epoch is not None else f"MedSAM {modo}"

    total_loss = total_bce = total_dice_loss = total_dice = total_iou = 0.0
    n = 0
    entrenar_encoder = train and modo == "decoder_encoder_parcial"

    for batch in tqdm(loader, desc=desc, leave=True):
        for sample in batch:
            gt = sample["gt_mask"].to(DEVICE)
            weight = sample["sample_weight"].to(DEVICE)
            if gt.ndim == 3:
                gt = gt.unsqueeze(0)

            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                    logits, _ = forward_medsam_diferenciable(
                        medsam,
                        sample["image"],
                        sample["box"],
                        entrenar_encoder=entrenar_encoder,
                    )
                    loss_bce = bce_loss_medsam(logits, gt)
                    loss_dice = dice_loss_from_logits_medsam(logits, gt)
                    loss = loss_bce + loss_dice
                    if train:
                        loss = loss * weight

                if train:
                    loss.backward()
                    optimizer.step()

            dice_val, iou_val = dice_iou_from_logits_medsam(logits.detach(), gt)
            total_loss += float(loss.detach().cpu())
            total_bce += float(loss_bce.detach().cpu())
            total_dice_loss += float(loss_dice.detach().cpu())
            total_dice += dice_val
            total_iou += iou_val
            n += 1

    return {
        "loss": total_loss / max(n, 1),
        "bce": total_bce / max(n, 1),
        "dice_loss": total_dice_loss / max(n, 1),
        "dice_bin": total_dice / max(n, 1),
        "iou_bin": total_iou / max(n, 1),
    }


# Entrena una variante, guarda el mejor checkpoint y exporta historial CSV.
def entrenar_variante_medsam_entrega(
    modo,
    train_loader,
    val_loader,
    epochs,
    paciencia,
    out_path,
    init_state_path=None,
):
    medsam = cargar_medsam_base_para_entrenar()
    if init_state_path is not None and Path(init_state_path).exists():
        state = torch_load_compatible(init_state_path, map_location=DEVICE)
        state = extraer_state_dict_medsam_entrenamiento(state)
        missing, unexpected = medsam.load_state_dict(state, strict=False)
        print(f"Inicializado desde {init_state_path} | faltantes={len(missing)} | inesperadas={len(unexpected)}")

    configurar_entrenabilidad_medsam(medsam, modo)
    optimizer = optimizador_medsam(medsam, modo)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    best_val = float("inf")
    best_state = None
    sin_mejora = 0
    hist = []

    for epoch in range(1, int(epochs) + 1):
        train_metrics = run_epoch_medsam_entrega(medsam, train_loader, optimizer=optimizer, modo=modo, epoch=epoch)
        val_metrics = run_epoch_medsam_entrega(medsam, val_loader, optimizer=None, modo=modo, epoch=epoch)
        scheduler.step(val_metrics["loss"])

        row = {
            "modo": modo,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_bce": train_metrics["bce"],
            "train_dice_loss": train_metrics["dice_loss"],
            "train_dice_bin": train_metrics["dice_bin"],
            "train_iou_bin": train_metrics["iou_bin"],
            "val_loss": val_metrics["loss"],
            "val_bce": val_metrics["bce"],
            "val_dice_loss": val_metrics["dice_loss"],
            "val_dice_bin": val_metrics["dice_bin"],
            "val_iou_bin": val_metrics["iou_bin"],
            "lr_min": min(g["lr"] for g in optimizer.param_groups),
            "lr_max": max(g["lr"] for g in optimizer.param_groups),
        }
        hist.append(row)
        print(
            f"{modo} epoch={epoch:02d} train_loss={row['train_loss']:.4f} "
            f"val_loss={row['val_loss']:.4f} val_dice={row['val_dice_bin']:.4f} val_iou={row['val_iou_bin']:.4f}"
        )

        if val_metrics["loss"] < best_val:
            best_val = val_metrics["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in medsam.state_dict().items()}
            sin_mejora = 0
            print("  -> nuevo mejor checkpoint")
        else:
            sin_mejora += 1
            print(f"  -> sin mejora ({sin_mejora}/{paciencia})")
            if sin_mejora >= paciencia:
                print("Early stopping MedSAM")
                break

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in medsam.state_dict().items()}

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model": best_state,
        "modo": modo,
        "config": {
            "frac_x": MEDSAM_TRAIN_FRAC_X,
            "frac_y": MEDSAM_TRAIN_FRAC_Y,
            "epochs_max": epochs,
            "paciencia": paciencia,
        },
    }, out_path)
    hist_df = pd.DataFrame(hist)
    hist_df.to_csv(RESULTADOS_DIR / f"historial_medsam_{modo}.csv", index=False)
    del medsam
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Checkpoint guardado:", out_path)
    return hist_df


# Secuencia de entrega: primero decoder, luego encoder parcial inicializado desde decoder.
if EJECUTAR_ENTRENAMIENTO_MEDSAM:
    train_loader_medsam, val_loader_medsam = crear_loaders_medsam_entrega()
    hist_medsam_decoder = entrenar_variante_medsam_entrega(
        modo="decoder",
        train_loader=train_loader_medsam,
        val_loader=val_loader_medsam,
        epochs=MEDSAM_DECODER_EPOCHS,
        paciencia=MEDSAM_DECODER_PATIENCIA,
        out_path=MEDSAM_DECODER_PATH,
    )
    display(hist_medsam_decoder.tail())

    hist_medsam_encoder_parcial = entrenar_variante_medsam_entrega(
        modo="decoder_encoder_parcial",
        train_loader=train_loader_medsam,
        val_loader=val_loader_medsam,
        epochs=MEDSAM_ENCODER_PARCIAL_EPOCHS,
        paciencia=MEDSAM_ENCODER_PARCIAL_PATIENCIA,
        out_path=MEDSAM_ENCODER_DECODER_PATH,
        init_state_path=MEDSAM_DECODER_PATH,
    )
    display(hist_medsam_encoder_parcial.tail())
else:
    print("Entrenamiento MedSAM desactivado. Se usaran checkpoints existentes si estan en las rutas configuradas.")


## 17. Variantes de MedSAM para la comparacion

Esta seccion carga las tres variantes usadas para decidir el modelo final. El baseline `medsam_general` permite medir cuanto aporta entrenar con el dataset local. Las otras dos variantes usan checkpoints generados por este notebook.

La mejor variante en `val` fue `medsam_decoder_encoder_parcial`. Esa fue la que se llevo a `test` como prueba final, evitando escoger el ganador mirando test.

<!-- codex-explicacion -->
Se comparan variantes de ajuste del modelo. La estrategia ganadora no depende solo del refinador, sino de la combinacion prompts + MedSAM.


In [ ]:
# Permite leer checkpoints guardados con distintas convenciones.
def extraer_state_dict_medsam(state):
    """Acepta checkpoints guardados como state_dict directo o envueltos en claves comunes."""
    if isinstance(state, dict):
        for key in ["model", "model_state_dict", "state_dict", "medsam"]:
            if key in state and isinstance(state[key], dict):
                return state[key]
    return state


# Carga una variante entrenada y la envuelve en SamPredictor para inferencia rapida.
def cargar_predictor_medsam_variante(variante):
    """Carga MedSAM base y, opcionalmente, pesos ya entrenados."""
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA no esta disponible. No se ejecuta MedSAM en CPU para evitar tiempos largos.")
    if not MEDSAM_CKPT_PATH.exists():
        raise FileNotFoundError(f"No existe el checkpoint base MedSAM: {MEDSAM_CKPT_PATH}")

    from segment_anything import sam_model_registry, SamPredictor

    device_medsam = torch.device("cuda")
    torch.cuda.empty_cache()

    model_medsam = sam_model_registry["vit_b"](checkpoint=str(MEDSAM_CKPT_PATH)).to(device_medsam)
    state_path = variante.get("state_path")
    carga_info = {
        "variante": variante["nombre"],
        "descripcion": variante["descripcion"],
        "state_path": str(state_path) if state_path else "",
        "missing": np.nan,
        "unexpected": np.nan,
    }

    if state_path:
        state_path = Path(state_path)
        if not state_path.exists():
            raise FileNotFoundError(f"No existe checkpoint de variante {variante['nombre']}: {state_path}")
        state = torch_load_compatible(state_path, map_location=device_medsam)
        state = extraer_state_dict_medsam(state)
        missing, unexpected = model_medsam.load_state_dict(state, strict=False)
        carga_info["missing"] = len(missing)
        carga_info["unexpected"] = len(unexpected)
        print(f"{variante['nombre']} cargado:", state_path, "| faltantes:", len(missing), "| inesperadas:", len(unexpected))
    else:
        print(f"{variante['nombre']} cargado desde MedSAM base:", MEDSAM_CKPT_PATH)

    model_medsam.eval()
    return SamPredictor(model_medsam), carga_info


# Lista final de modelos que entra al comparativo con las mismas cajas NN-SAM.
MEDSAM_VARIANTES_NN_SAM = [
    {
        "nombre": "medsam_decoder_encoder_parcial",
        "descripcion": "Modelo final: MedSAM con decoder y ultimo bloque del encoder ajustados sobre prompts VertebraPrompt-Net",
        "state_path": MEDSAM_ENCODER_DECODER_PATH,
    },
]


## 18. Prompts automaticos usados por MedSAM

MedSAM recibe todas las cajas detectadas por VertebraPrompt-Net. Para la metrica estricta se puede aplicar el shift estimado; para la visualizacion se conserva toda la columna segmentada.

Tambien se probaron expansiones de caja (`pad_03_04`, `pad_04_06`). Aunque antes parecia razonable dar mas contexto, en esta version gano `sin_pad`. Eso indica que expandir la caja puede meter costillas, tejido vecino u otra vertebra, reduciendo la precision de MedSAM.

<!-- codex-explicacion -->
Esta seccion confirma que la evaluacion final usa prompts automaticos refinados. Es el punto que diferencia el pipeline real de una prueba con ground truth.


In [ ]:
# Genera prompts automaticos y, opcionalmente, aplica shift solo para evaluacion estricta.
def obtener_prompts_nn_sam_para_medsam(split, patient_id, usar_shift=True):
    """Devuelve prompts crudos/refinados y prompts con shift para metrica estricta."""
    img, mask_gt, prompts_raw, _, _, _ = generar_prompts_nn(split, patient_id, modo_etiquetado=ETIQUETADO_MODO)

    if APLICAR_BOX_REFINER_A_MEDSAM and globals().get("box_refiner_model_final") is not None:
        prompts_raw, _ = refinar_prompts_con_box_refiner(prompts_raw, img, globals().get("box_refiner_model_final"))

    best_shift = 0
    prompts_eval = prompts_raw
    if usar_shift:
        best_shift, _, _ = evaluar_shifts_anatomicos(prompts_raw, mask_gt)
        prompts_eval, _ = relabel_prompts_por_shift(prompts_raw, best_shift)
    return img, mask_gt, prompts_raw, prompts_eval, best_shift


def centro_y_bbox(info):
    x0, y0, x1, y1 = [float(v) for v in info["bbox_xyxy"]]
    return (y0 + y1) / 2


# Etiquetado visual top-down/bottom-up para inspeccionar imagenes con GT parcial.
def asignar_labels_por_orden(prompts_raw, modo="top_down"):
    """Asigna etiquetas anatomicas solo para visualizacion segun orden vertical."""
    items = sorted(prompts_raw.items(), key=lambda kv: centro_y_bbox(kv[1]))
    n = min(len(items), N_CLASES)
    if modo == "bottom_up":
        labels = CLASES_OBJETIVO[N_CLASES - n:]
    else:
        labels = CLASES_OBJETIVO[:n]
    return {raw_label: labels[i] for i, (raw_label, _) in enumerate(items[:n])}


def construir_mascara_semantica_desde_preds(pred_masks, labels_forzados=None):
    """Construye una mascara semantica desde mascaras binarias por vertebra."""
    if not pred_masks:
        return None
    first = next(iter(pred_masks.values()))
    mask_sem = np.zeros(first.shape, dtype=np.uint8)
    for vertebra, pred in sorted(pred_masks.items(), key=lambda kv: VERTEBRA_TO_ID.get(kv[0], 999)):
        label = labels_forzados.get(vertebra, vertebra) if labels_forzados else vertebra
        id_v = VERTEBRA_TO_ID.get(label, 0)
        if id_v > 0:
            mask_sem[pred.astype(bool)] = id_v
    return mask_sem


# Metrica flexible: para cada vertebra GT busca la mejor mascara segmentada.
def evaluar_predicciones_flexibles(pred_masks, mask_gt, vertebras_gt):
    """Para cada vertebra GT busca la mejor mascara predicha entre todas las cajas segmentadas."""
    filas = []
    for gt_v in vertebras_gt:
        gt = mask_binaria_vertebra(mask_gt, gt_v).astype(bool)
        mejor = None
        mejor_key = None
        for pred_v, pred in pred_masks.items():
            met = dice_iou_binario(pred, gt)
            key = (met["dice"], met["iou"])
            if mejor is None or key > mejor_key:
                pred_id = VERTEBRA_TO_ID.get(pred_v, np.nan)
                gt_id = VERTEBRA_TO_ID.get(gt_v, np.nan)
                mejor = {
                    "gt_vertebra": gt_v,
                    "gt_id": gt_id,
                    "prompt_flexible_vertebra": pred_v,
                    "prompt_flexible_id": pred_id,
                    "dice_flexible": float(met["dice"]) if np.isfinite(met["dice"]) else np.nan,
                    "iou_flexible": float(met["iou"]) if np.isfinite(met["iou"]) else np.nan,
                    "pix_gt": int(met["pix_gt"]),
                    "pix_pred_flexible": int(met["pix_pred"]),
                    "flexible_misma_etiqueta": bool(pred_v == gt_v),
                    "desfase_id_flexible": int(pred_id - gt_id) if np.isfinite(pred_id) and np.isfinite(gt_id) else np.nan,
                }
                mejor_key = key
        if mejor is not None:
            filas.append(mejor)
    return pd.DataFrame(filas)


# Segmenta un paciente completo con una variante MedSAM y guarda tablas/figuras.
def segmentar_paciente_con_predictor(
    predictor,
    variante_nombre,
    patient_id="S_187",
    split="val",
    usar_shift=True,
    prompt_mode="box_only",
    max_vertebras=None,
    guardar=True,
    bbox_expansion_nombre="sin_pad",
    bbox_frac_x=0.0,
    bbox_frac_y=0.0,
):
    img, mask_gt, prompts_raw, prompts_eval, best_shift = obtener_prompts_nn_sam_para_medsam(split, patient_id, usar_shift=usar_shift)
    predictor.set_image(img)

    vertebras_gt = vertebras_gt_eval(mask_gt)
    labels_gt_set = set(vertebras_gt)
    pred_masks = {}
    detalles = []

    labels_top_down = asignar_labels_por_orden(prompts_raw, modo="top_down")
    labels_bottom_up = asignar_labels_por_orden(prompts_raw, modo="bottom_up")

    # Relaciona cada caja cruda con la etiqueta corregida por shift usada para metrica estricta.
    raw_to_eval = {}
    for eval_label, info in prompts_eval.items():
        raw_label = info.get("vertebra_original", eval_label)
        raw_to_eval[raw_label] = eval_label

    # Se segmentan todas las cajas originales para visualizar toda la columna propuesta.
    items = sorted(prompts_raw.items(), key=lambda kv: centro_y_bbox(kv[1]))
    if max_vertebras is not None:
        items = items[:int(max_vertebras)]

    t0 = time.perf_counter()
    for raw_label, info in tqdm(items, desc=f"{variante_nombre} {patient_id}", leave=False):
        bbox = [int(v) for v in info["bbox_xyxy"]]
        bbox_prompt = expandir_bbox_xyxy_frac(bbox, img.shape, bbox_frac_x, bbox_frac_y)
        kwargs = preparar_prompt_medsam(img, bbox, modo=prompt_mode, bbox_frac_x=bbox_frac_x, bbox_frac_y=bbox_frac_y)
        masks, scores, logits = predictor.predict(**kwargs)
        pred = masks[0].astype(bool)
        pred_masks[raw_label] = pred

        eval_label = raw_to_eval.get(raw_label, raw_label)
        if eval_label in labels_gt_set:
            gt = mask_binaria_vertebra(mask_gt, eval_label).astype(bool)
            met = dice_iou_binario(pred, gt)
            estado_eval = "evaluado_gt"
        else:
            met = {"dice": np.nan, "iou": np.nan, "pix_pred": int(pred.sum()), "pix_gt": 0}
            estado_eval = "prompt_extra_sin_gt"

        detalles.append({
            "variante": variante_nombre,
            "patient_id": patient_id,
            "tipo_real": "escoliosis" if str(patient_id).startswith("S_") else "normal",
            "split": split,
            "vertebra": eval_label,
            "vertebra_original": raw_label,
            "visual_label_top_down": labels_top_down.get(raw_label, ""),
            "visual_label_bottom_up": labels_bottom_up.get(raw_label, ""),
            "shift_anatomico": int(best_shift),
            "prompt_mode": prompt_mode,
            "bbox_expansion": bbox_expansion_nombre,
            "bbox_frac_x": float(bbox_frac_x),
            "bbox_frac_y": float(bbox_frac_y),
            "estado_eval": estado_eval,
            "score_medsam": float(scores[0]),
            "dice_estricto": float(met["dice"]) if np.isfinite(met["dice"]) else np.nan,
            "iou_estricto": float(met["iou"]) if np.isfinite(met["iou"]) else np.nan,
            "pix_gt": int(met["pix_gt"]),
            "pix_pred": int(met["pix_pred"]),
            "bbox_xyxy": bbox,
            "bbox_prompt_xyxy": [float(v) for v in bbox_prompt],
        })

    tiempo_total = time.perf_counter() - t0
    df_prompts = pd.DataFrame(detalles)
    df_prompts["n_gt"] = len(vertebras_gt)
    df_prompts["n_segmentadas_total"] = len(items)
    df_prompts["tiempo_total_s"] = tiempo_total
    df_prompts["tiempo_por_vertebra_s"] = tiempo_total / max(len(df_prompts), 1)

    df_flexible = evaluar_predicciones_flexibles(pred_masks, mask_gt, vertebras_gt)
    if not df_flexible.empty:
        df_flexible["variante"] = variante_nombre
        df_flexible["patient_id"] = patient_id
        df_flexible["tipo_real"] = "escoliosis" if str(patient_id).startswith("S_") else "normal"
        df_flexible["split"] = split
        df_flexible["prompt_mode"] = prompt_mode
        df_flexible["bbox_expansion"] = bbox_expansion_nombre
        df_flexible["bbox_frac_x"] = float(bbox_frac_x)
        df_flexible["bbox_frac_y"] = float(bbox_frac_y)
        df_flexible["shift_anatomico"] = int(best_shift)
        df_flexible["n_gt"] = len(vertebras_gt)
        df_flexible["n_segmentadas_total"] = len(items)

    mascara_sem_original = construir_mascara_semantica_desde_preds(pred_masks)
    mascara_sem_top_down = construir_mascara_semantica_desde_preds(pred_masks, labels_forzados=labels_top_down)
    mascara_sem_bottom_up = construir_mascara_semantica_desde_preds(pred_masks, labels_forzados=labels_bottom_up)

    labels_flex = {}
    if not df_flexible.empty:
        for _, row in df_flexible.iterrows():
            labels_flex[row["prompt_flexible_vertebra"]] = row["gt_vertebra"]
    pred_masks_flex = {
        row["prompt_flexible_vertebra"]: pred_masks[row["prompt_flexible_vertebra"]]
        for _, row in df_flexible.iterrows()
        if row["prompt_flexible_vertebra"] in pred_masks
    } if not df_flexible.empty else {}
    mascara_sem_flexible = construir_mascara_semantica_desde_preds(pred_masks_flex, labels_forzados=labels_flex)

    if guardar and MEDSAM_GUARDAR_FIGURAS:
        out_base = f"medsam_nn_sam_{split}_{patient_id}_{variante_nombre}_{prompt_mode}_{bbox_expansion_nombre}"
        df_prompts.to_csv(NN_SAM_DIR / f"{out_base}_prompts.csv", index=False)
        df_flexible.to_csv(NN_SAM_DIR / f"{out_base}_flexible.csv", index=False)

        fig, ax = plt.subplots(1, 6, figsize=(34, 7))
        ax[0].imshow(img)
        ax[0].set_title("Radiografia")
        ax[1].imshow(mask_gt, cmap="nipy_spectral")
        ax[1].set_title(f"GT disponible | n={len(vertebras_gt)}")
        ax[2].imshow(mascara_sem_original, cmap="nipy_spectral")
        ax[2].set_title(f"Original/shift | n={len(items)}")
        ax[3].imshow(mascara_sem_top_down, cmap="nipy_spectral")
        ax[3].set_title("Visual top-down")
        ax[4].imshow(mascara_sem_bottom_up, cmap="nipy_spectral")
        ax[4].set_title("Visual bottom-up")
        ax[5].imshow(mascara_sem_flexible if mascara_sem_flexible is not None else np.zeros_like(mask_gt), cmap="nipy_spectral")
        ax[5].set_title("Mejor mascara flexible vs GT")
        for a in ax:
            a.axis("off")
        plt.tight_layout()
        plt.savefig(NN_SAM_DIR / f"{out_base}.png", dpi=140, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    return df_prompts, df_flexible


## 19. Comparativo MedSAM con prompts automaticos

Este bloque compara variantes MedSAM usando exactamente las mismas cajas VertebraPrompt-Net. Asi se evita confundir mejoras de segmentacion con cambios en deteccion.

En validacion, la mejor configuracion fue:

- Modelo: `medsam_decoder_encoder_parcial`
- Prompt: `box_only`
- Caja: `sin_pad`

Resultado en validacion para esa ruta:

- Dice estricto aproximado: `0.575`
- IoU estricto aproximado: `0.483`
- Dice flexible aproximado: `0.753`
- IoU flexible aproximado: `0.632`

La diferencia entre estricto y flexible confirma que MedSAM segmenta bien muchas vertebras, pero el nombramiento anatomico sigue siendo el factor mas dificil.

<!-- codex-explicacion -->
La tabla final muestra que BoxRefiner mejora tanto metrica estricta como flexible frente a NN-SAM robusto.


In [ ]:
# Ejecuta el comparativo final: variantes MedSAM x expansiones de bbox x pacientes trazadores.
def comparar_variantes_medsam_nn_sam(
    patient_ids=None,
    split="val",
    variantes=None,
    usar_shift=True,
    prompt_mode="box_only",
    max_vertebras=None,
    expansiones_bbox=None,
):
    patient_ids = patient_ids or MEDSAM_COMPARAR_PATIENT_IDS
    variantes = variantes or MEDSAM_VARIANTES_NN_SAM
    expansiones_bbox = expansiones_bbox or MEDSAM_EXPANSIONES_BBOX

    detalles_prompts = []
    detalles_flexibles = []
    cargas = []
    for variante in variantes:
        predictor, carga_info = cargar_predictor_medsam_variante(variante)
        cargas.append(carga_info)
        for expansion in expansiones_bbox:
            for pid in patient_ids:
                df_prompts_pid, df_flex_pid = segmentar_paciente_con_predictor(
                    predictor=predictor,
                    variante_nombre=variante["nombre"],
                    patient_id=pid,
                    split=split,
                    usar_shift=usar_shift,
                    prompt_mode=prompt_mode,
                    max_vertebras=max_vertebras,
                    guardar=True,
                    bbox_expansion_nombre=expansion["nombre"],
                    bbox_frac_x=expansion["frac_x"],
                    bbox_frac_y=expansion["frac_y"],
                )
                detalles_prompts.append(df_prompts_pid)
                detalles_flexibles.append(df_flex_pid)
        del predictor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df_prompts = pd.concat(detalles_prompts, ignore_index=True) if detalles_prompts else pd.DataFrame()
    df_flexible = pd.concat(detalles_flexibles, ignore_index=True) if detalles_flexibles else pd.DataFrame()
    df_cargas = pd.DataFrame(cargas)

    if df_prompts.empty:
        return pd.DataFrame(), pd.DataFrame(), df_prompts, df_flexible, df_cargas

    # Solo las mascaras con GT disponible entran a la metrica estricta.
    df_eval = df_prompts[df_prompts["estado_eval"] == "evaluado_gt"].copy()

    df_resumen_paciente = (
        df_eval
        .groupby(["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"], as_index=False)
        .agg(
            n_gt=("n_gt", "max"),
            n_eval_estricto=("vertebra", "count"),
            dice_estricto_promedio=("dice_estricto", "mean"),
            iou_estricto_promedio=("iou_estricto", "mean"),
            score_medsam_promedio=("score_medsam", "mean"),
            tiempo_total_s=("tiempo_total_s", "max"),
            tiempo_por_vertebra_s=("tiempo_por_vertebra_s", "mean"),
            shift_anatomico=("shift_anatomico", "max"),
        )
    )

    seg_counts = (
        df_prompts
        .groupby(["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"], as_index=False)
        .agg(n_segmentadas_total=("vertebra", "count"))
    )
    df_resumen_paciente = df_resumen_paciente.merge(
        seg_counts,
        on=["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"],
        how="left",
    )

    if not df_flexible.empty:
        flex_res = (
            df_flexible
            .groupby(["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"], as_index=False)
            .agg(
                n_eval_flexible=("gt_vertebra", "count"),
                dice_flexible_promedio=("dice_flexible", "mean"),
                iou_flexible_promedio=("iou_flexible", "mean"),
                n_flexible_misma_etiqueta=("flexible_misma_etiqueta", "sum"),
                desfase_flexible_promedio=("desfase_id_flexible", "mean"),
            )
        )
        df_resumen_paciente = df_resumen_paciente.merge(
            flex_res,
            on=["variante", "split", "patient_id", "tipo_real", "prompt_mode", "bbox_expansion"],
            how="left",
        )

    # Resumen por grupo clinico: normal vs escoliosis.
    df_resumen_tipo = (
        df_resumen_paciente
        .groupby(["variante", "tipo_real", "prompt_mode", "bbox_expansion"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            n_eval_estricto=("n_eval_estricto", "sum"),
            n_segmentadas_total=("n_segmentadas_total", "sum"),
            dice_estricto_promedio=("dice_estricto_promedio", "mean"),
            iou_estricto_promedio=("iou_estricto_promedio", "mean"),
            dice_flexible_promedio=("dice_flexible_promedio", "mean"),
            iou_flexible_promedio=("iou_flexible_promedio", "mean"),
            tiempo_por_vertebra_s=("tiempo_por_vertebra_s", "mean"),
        )
    )

    # Resumen global para escoger variante y expansion candidata.
    df_global = (
        df_resumen_paciente
        .groupby(["variante", "prompt_mode", "bbox_expansion"], as_index=False)
        .agg(
            pacientes=("patient_id", "nunique"),
            n_eval_estricto=("n_eval_estricto", "sum"),
            n_segmentadas_total=("n_segmentadas_total", "sum"),
            dice_estricto_promedio=("dice_estricto_promedio", "mean"),
            iou_estricto_promedio=("iou_estricto_promedio", "mean"),
            dice_flexible_promedio=("dice_flexible_promedio", "mean"),
            iou_flexible_promedio=("iou_flexible_promedio", "mean"),
            tiempo_por_vertebra_s=("tiempo_por_vertebra_s", "mean"),
        )
        .sort_values(["dice_flexible_promedio", "dice_estricto_promedio"], ascending=False)
    )

    df_prompts.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_detalle_prompts.csv", index=False)
    df_flexible.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_detalle_flexible.csv", index=False)
    df_resumen_paciente.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_resumen_paciente.csv", index=False)
    df_resumen_tipo.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_resumen_tipo.csv", index=False)
    df_global.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_resumen_global.csv", index=False)
    df_cargas.to_csv(NN_SAM_DIR / f"comparativo_medsam_nn_sam_{split}_cargas.csv", index=False)

    display(df_global)
    display(df_resumen_tipo)
    display(df_resumen_paciente)
    display(df_cargas)
    return df_global, df_resumen_tipo, df_prompts, df_flexible, df_cargas


# Lanza la evaluacion final despues de entrenar/cargar las variantes.
if EJECUTAR_COMPARATIVO_MEDSAM:
    df_medsam_variantes_global, df_medsam_variantes_tipo, df_medsam_variantes_prompts, df_medsam_variantes_flexible, df_medsam_variantes_cargas = comparar_variantes_medsam_nn_sam(
        patient_ids=MEDSAM_COMPARAR_PATIENT_IDS,
        split=MEDSAM_DEMO_SPLIT,
        usar_shift=MEDSAM_COMPARAR_USAR_SHIFT,
        prompt_mode=MEDSAM_COMPARAR_PROMPT_MODE,
        max_vertebras=MEDSAM_DEMO_MAX_VERTEBRAS,
        expansiones_bbox=MEDSAM_EXPANSIONES_BBOX,
    )
elif EJECUTAR_MEDSAM_BASICO:
    df_medsam_demo = comparar_variantes_medsam_nn_sam(
        patient_ids=[MEDSAM_DEMO_PATIENT_ID],
        split=MEDSAM_DEMO_SPLIT,
        variantes=[MEDSAM_VARIANTES_NN_SAM[1]],
        usar_shift=MEDSAM_DEMO_USAR_SHIFT,
        prompt_mode=MEDSAM_COMPARAR_PROMPT_MODE,
        max_vertebras=MEDSAM_DEMO_MAX_VERTEBRAS,
        expansiones_bbox=MEDSAM_EXPANSIONES_BBOX,
    )[2]
else:
    print("MedSAM no se ejecuto. Activa EJECUTAR_COMPARATIVO_MEDSAM=True para comparar variantes.")


## 20. Lectura de resultados esperada

Esta version debe compararse contra dos referencias:

1. El baseline ganador anterior: buenas cajas y buen MedSAM.
2. El VertebraPrompt-Net fallido: mostro que usar `class_heat` para escoger centros destruye la localizacion.

La rama anatomica se entrena solo en centros GT para no perturbar la localizacion. La senal minima de exito aqui es que las metricas de prompts vuelvan a quedar cerca del baseline anterior. Si eso ocurre, la rama anatomica queda disponible como diagnostico o como correccion posterior. Si el Dice flexible vuelve a caer, entonces incluso el entrenamiento auxiliar esta perturbando demasiado la red y se debe separar por completo en otra red de nombres/crops.

Archivos principales:

1. `vertebraprompt_net_auxiliar_best.pt`: detector ganador + rama anatomica auxiliar.
2. `historial_entrenamiento_vertebraprompt_auxiliar.csv`: entrenamiento de la rama auxiliar.
3. `nn_sam_val_resumen.csv` y `nn_sam_test_resumen.csv`: calidad de prompts antes de MedSAM.
4. `FINAL_TEST_medsam_nn_sam_resumen_global.csv`: resultado final con MedSAM.

<!-- codex-explicacion -->
La lectura correcta es que la mejora no es gigantesca, pero si consistente: aproximadamente +1.3 en Dice estricto y +1.9 en Dice flexible.


## 21. Test final con la mejor estrategia

Despues de elegir la mejor configuracion en validacion, se evalua una sola ruta en `test`:

- `medsam_decoder_encoder_parcial`
- `box_only`
- `sin_pad`

Esto es metodologicamente importante. Si se probaran todas las variantes en `test` y luego se escogiera la mejor, el test dejaria de ser una evaluacion final limpia. Aqui `test` se usa para confirmar si la estrategia elegida en `val` generaliza.

Esta celda no reentrena nada. Solo carga el checkpoint ya entrenado y calcula resultados finales sobre el split `test`.

<!-- codex-explicacion -->
El test final se usa para reportar la estrategia ganadora ya congelada. No debe usarse para seguir ajustando hiperparametros.


In [ ]:
# ============================================================
# TEST FINAL CON LA MEJOR CONFIGURACION EN VAL
# ============================================================
# Objetivo:
# Evaluar en test la ruta que mejor funciono en validacion:
#   - MedSAM decoder + encoder parcial
#   - prompts automaticos NN-SAM
#   - box_only
#   - sin expansion adicional de caja
#
# Esta celda NO reentrena. Solo carga el checkpoint ya entrenado
# y calcula metricas/figuras sobre test.

EJECUTAR_MEDSAM_TEST_FINAL = True
MEDSAM_TEST_USAR_SHIFT = True
MEDSAM_TEST_PROMPT_MODE = "box_only"
MEDSAM_TEST_MAX_VERTEBRAS = None
MEDSAM_TEST_GUARDAR_FIGURAS = True

# Usar todo el split test. Si quieres una prueba rapida primero,
# reemplaza esta lista por algunos IDs concretos.
MEDSAM_TEST_PATIENT_IDS = sorted(PROMPTS_DICC["test"].keys())

# Mejor configuracion segun val.
MEDSAM_TEST_VARIANTES = [
    {
        "nombre": "medsam_decoder_encoder_parcial",
        "descripcion": "MedSAM con decoder y ultimo bloque del encoder ajustados en este notebook",
        "state_path": MEDSAM_ENCODER_DECODER_PATH,
    }
]

MEDSAM_TEST_EXPANSIONES = [
    {"nombre": "sin_pad", "frac_x": 0.00, "frac_y": 0.00},
]

if EJECUTAR_MEDSAM_TEST_FINAL:
    # Reutiliza las funciones ya definidas en el notebook.
    # No vuelve a entrenar cajas ni MedSAM.
    df_medsam_test_global, df_medsam_test_tipo, df_medsam_test_prompts, df_medsam_test_flexible, df_medsam_test_cargas = comparar_variantes_medsam_nn_sam(
        patient_ids=MEDSAM_TEST_PATIENT_IDS,
        split="test",
        variantes=MEDSAM_TEST_VARIANTES,
        usar_shift=MEDSAM_TEST_USAR_SHIFT,
        prompt_mode=MEDSAM_TEST_PROMPT_MODE,
        max_vertebras=MEDSAM_TEST_MAX_VERTEBRAS,
        expansiones_bbox=MEDSAM_TEST_EXPANSIONES,
    )

    # Guardar con nombres explicitos para no confundir con val.
    df_medsam_test_global.to_csv(NN_SAM_DIR / "FINAL_TEST_medsam_nn_sam_resumen_global.csv", index=False)
    df_medsam_test_tipo.to_csv(NN_SAM_DIR / "FINAL_TEST_medsam_nn_sam_resumen_tipo.csv", index=False)
    df_medsam_test_prompts.to_csv(NN_SAM_DIR / "FINAL_TEST_medsam_nn_sam_detalle_prompts.csv", index=False)
    df_medsam_test_flexible.to_csv(NN_SAM_DIR / "FINAL_TEST_medsam_nn_sam_detalle_flexible.csv", index=False)

    display(df_medsam_test_global)
    display(df_medsam_test_tipo)


## Conclusion esperada de BoxRefiner

Esta prueba debe mostrar si corregir localmente las cajas mejora el rendimiento flexible y la segmentacion final de MedSAM. Si sube `bbox_iou_flexible`, baja `center_error_flexible_px` y mejora el Dice flexible final, BoxRefiner es una ruta prometedora. Si baja el estricto, se puede usar solo como opcion para segmentacion flexible, no para etiquetado anatomico.

<!-- codex-explicacion -->
La conclusion es que el cuello de botella estaba parcialmente en la calidad del prompt. BoxRefiner no elimina todos los errores anatomicos, pero mejora el promedio global.


<!-- codex-cierre-etapa -->
## Cierre de etapa y proyecci?n

Esta etapa re?ne la estrategia ganadora por m?tricas promedio en test: VertebraPrompt + BoxRefiner + MedSAM. La mejora frente a NN-SAM robusto no aparece por cambiar todo el pipeline, sino por refinar mejor los prompts vertebrales antes de llamar a MedSAM.

La conclusi?n para el informe no debe ser que el problema qued? completamente cerrado. La mejora es relevante, pero todav?a queda una limitaci?n clara: la asignaci?n anat?mica estricta puede fallar aunque la regi?n segmentada sea razonable. Por eso una etapa futura deber?a centrarse en mejorar la consistencia anat?mica, validar con m?s datos externos o estudiar una red espec?fica de nombramiento vertebral integrada al detector.

**Siguiente etapa sugerida:** optimizaci?n de consistencia anat?mica y validaci?n externa, no una nueva b?squeda amplia de cajas.
